# DNN

In [8]:
import os
import sys
import time
import socket
import platform
import hashlib

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import psutil
from torchvision import datasets, transforms


# ============================================================
# 0. RUN CONFIGURATION
# ============================================================

# Device configuration:
# "auto" = CUDA if available, otherwise CPU
# "cpu"  = force CPU
# "cuda" = force CUDA if available
DEVICE_MODE = "auto"
NUM_TEST_SAMPLES = 10
# Number of MNIST test samples to run.
# Use None for all 10,000 samples.
# Use a small number like 10 for debugging.

# Save outputs
SAVE_EXCEL = True
SAVE_CSV = False

# Measure energy for all four agent steps.
# Warning: very short steps may have noisy CodeCarbon values.
AGENT_STEP_CONFIG = {
    "[AGENT INIT]": {
        "measure_energy": True,
        "description": "Initialize the AI agent and load model metadata."
    },
    "[AGENT TASK RECEIVED]": {
        "measure_energy": True,
        "description": "Receive the user task."
    },
    "[AGENT PLANNING]": {
        "measure_energy": True,
        "description": "Check task type and build the execution plan."
    },
    "[AGENT ACTION]": {
        "measure_energy": True,
        "description": "Run per-sample MNIST classification."
    },
}


def resolve_device():
    if DEVICE_MODE.lower() == "cpu":
        return torch.device("cpu")

    if DEVICE_MODE.lower() == "cuda":
        if torch.cuda.is_available():
            return torch.device("cuda")
        print("CUDA requested but not available. Falling back to CPU.")
        return torch.device("cpu")

    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE_EXEC = resolve_device()


# ============================================================
# 1. Safe optional imports
# ============================================================

# ---- CodeCarbon ----
try:
    from codecarbon import EmissionsTracker
    import codecarbon
    CODECARBON_AVAILABLE = True
    CODECARBON_VERSION = codecarbon.__version__
except Exception:
    EmissionsTracker = None
    CODECARBON_AVAILABLE = False
    CODECARBON_VERSION = "unavailable"
    print("CodeCarbon not available. Energy values will be set to 0.")

# ---- pynvml ----
try:
    import pynvml
    pynvml.nvmlInit()
    NVML_AVAILABLE = True
    NVML_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0) if torch.cuda.is_available() else None
except Exception:
    NVML_AVAILABLE = False
    NVML_HANDLE = None

# ---- py-cpuinfo ----
try:
    import cpuinfo
    _CPU_INFO = cpuinfo.get_cpu_info()
    CPU_MODEL_NAME = _CPU_INFO.get("brand_raw", "Unknown")
    CPU_ARCH = _CPU_INFO.get("arch", platform.machine())
    CPU_TDP_W = None
except Exception:
    CPU_MODEL_NAME = "Unknown"
    CPU_ARCH = platform.machine()
    CPU_TDP_W = None

# ---- fvcore: FLOPs ----
try:
    from fvcore.nn import FlopCountAnalysis
    FVCORE_AVAILABLE = True
except Exception:
    FlopCountAnalysis = None
    FVCORE_AVAILABLE = False

# ---- sklearn metrics ----
try:
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    SKLEARN_AVAILABLE = True
except Exception:
    accuracy_score = None
    precision_recall_fscore_support = None
    SKLEARN_AVAILABLE = False
    print("sklearn not available. Final precision/recall/F1 will be computed with basic accuracy only.")


# ============================================================
# 2. OS / environment helpers
# ============================================================

def get_os_full_name():
    system = platform.system()
    architecture = platform.machine()

    if system == "Windows":
        try:
            import winreg
            key = winreg.OpenKey(
                winreg.HKEY_LOCAL_MACHINE,
                r"SOFTWARE\Microsoft\Windows NT\CurrentVersion"
            )
            product_name = winreg.QueryValueEx(key, "ProductName")[0]
            display_version = winreg.QueryValueEx(key, "DisplayVersion")[0]
            current_build = winreg.QueryValueEx(key, "CurrentBuild")[0]
            return f"{product_name} {display_version} Build {current_build} {architecture}"
        except Exception:
            return f"Windows {platform.release()} {architecture}"

    if system == "Linux":
        os_info = {}
        try:
            with open("/etc/os-release", "r", encoding="utf-8") as file:
                for line in file:
                    if "=" in line:
                        key, value = line.strip().split("=", 1)
                        os_info[key] = value.strip('"')
        except Exception:
            pass

        pretty_name = os_info.get("PRETTY_NAME")
        name = os_info.get("NAME")
        version = os_info.get("VERSION")
        version_id = os_info.get("VERSION_ID")
        distro_id = os_info.get("ID")

        if pretty_name:
            return f"{pretty_name} {architecture}"
        if name and version:
            return f"{name} {version} {architecture}"
        if name and version_id:
            return f"{name} {version_id} {architecture}"
        if distro_id:
            return f"{distro_id} {platform.release()} {architecture}"
        return f"Linux {platform.release()} {architecture}"

    if system == "Darwin":
        return f"macOS {platform.mac_ver()[0]} {architecture}"

    return f"{system} {platform.release()} {architecture}"


TORCH_VERSION = torch.__version__
PYTHON_VERSION = sys.version.split()[0]
OS_NAME = platform.system()
OS_VERSION = platform.version()
OS_ARCHITECTURE = platform.machine()
OS_FULL_NAME = get_os_full_name()
SYSTEM_RAM_TOTAL_GB = round(psutil.virtual_memory().total / (1024 ** 3), 2)
CPU_CORE_COUNT = psutil.cpu_count(logical=False)
CPU_THREAD_COUNT = psutil.cpu_count(logical=True)


# ============================================================
# 3. Stable device ID
# ============================================================

def make_stable_device_id():
    raw = f"{socket.gethostname()}-{platform.system()}-{platform.machine()}-{CPU_MODEL_NAME}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


DEVICE_UUID = make_stable_device_id()
DEVICE_SHORT = DEVICE_UUID[:8]


# ============================================================
# 4. GPU static info
# ============================================================

def _get_cuda_driver_version():
    if not NVML_AVAILABLE:
        return None

    try:
        driver = pynvml.nvmlSystemGetDriverVersion()
        return driver.decode("utf-8") if isinstance(driver, bytes) else driver
    except Exception:
        return None


CUDA_DRIVER_VERSION = _get_cuda_driver_version()


def _get_gpu_static():
    defaults = {
        "gpu_power_limit_w": None,
        "gpu_driver_version": CUDA_DRIVER_VERSION,
        "gpu_memory_total_mb": None,
        "gpu_compute_capability": None,
    }

    if not NVML_AVAILABLE or NVML_HANDLE is None:
        return defaults

    try:
        power_limit_mw = pynvml.nvmlDeviceGetPowerManagementLimit(NVML_HANDLE)
        mem_info = pynvml.nvmlDeviceGetMemoryInfo(NVML_HANDLE)
        cc_major, cc_minor = pynvml.nvmlDeviceGetCudaComputeCapability(NVML_HANDLE)

        return {
            "gpu_power_limit_w": round(power_limit_mw / 1000.0, 1),
            "gpu_driver_version": CUDA_DRIVER_VERSION,
            "gpu_memory_total_mb": round(mem_info.total / (1024 ** 2), 2),
            "gpu_compute_capability": f"{cc_major}.{cc_minor}",
        }
    except Exception:
        return defaults


GPU_STATIC = _get_gpu_static()


# ============================================================
# 5. Per-sample device helpers
# ============================================================

def get_hostname():
    return socket.gethostname()


def get_gpu_name():
    return torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU"


def get_cpu_usage():
    return psutil.cpu_percent(interval=None)


def get_ram_usage():
    return psutil.virtual_memory().percent


def get_cpu_freq():
    freq = psutil.cpu_freq()
    return round(freq.current, 2) if freq else None


def get_memory_footprint_mb():
    process = psutil.Process(os.getpid())
    return round(process.memory_info().rss / (1024 * 1024), 4)


def get_gpu_metrics():
    null = {
        "gpu_power_draw_w": None,
        "gpu_utilization_pct": None,
        "gpu_temp_c": None,
        "gpu_memory_used_mb": None,
        "gpu_sm_clock_mhz": None,
        "gpu_memory_clock_mhz": None,
    }

    if not NVML_AVAILABLE or NVML_HANDLE is None:
        return null

    try:
        power_mw = pynvml.nvmlDeviceGetPowerUsage(NVML_HANDLE)
        util = pynvml.nvmlDeviceGetUtilizationRates(NVML_HANDLE)
        temp = pynvml.nvmlDeviceGetTemperature(NVML_HANDLE, pynvml.NVML_TEMPERATURE_GPU)
        mem_info = pynvml.nvmlDeviceGetMemoryInfo(NVML_HANDLE)
        sm_clock = pynvml.nvmlDeviceGetClockInfo(NVML_HANDLE, pynvml.NVML_CLOCK_SM)
        mem_clock = pynvml.nvmlDeviceGetClockInfo(NVML_HANDLE, pynvml.NVML_CLOCK_MEM)

        return {
            "gpu_power_draw_w": round(power_mw / 1000.0, 2),
            "gpu_utilization_pct": util.gpu,
            "gpu_temp_c": temp,
            "gpu_memory_used_mb": round(mem_info.used / (1024 ** 2), 2),
            "gpu_sm_clock_mhz": sm_clock,
            "gpu_memory_clock_mhz": mem_clock,
        }
    except Exception:
        return null


def get_cpu_temp():
    try:
        temps = psutil.sensors_temperatures()
        if not temps:
            return None

        for key in ("coretemp", "k10temp", "cpu_thermal", "acpitz"):
            if key in temps:
                values = [e.current for e in temps[key] if e.current and e.current > 0]
                if values:
                    return round(sum(values) / len(values), 1)
    except Exception:
        pass

    return None


def get_cpu_power_draw_w():
    rapl_path = "/sys/class/powercap/intel-rapl/intel-rapl:0/energy_uj"

    try:
        if os.path.exists(rapl_path):
            with open(rapl_path) as f:
                e1 = int(f.read().strip())

            time.sleep(0.1)

            with open(rapl_path) as f:
                e2 = int(f.read().strip())

            return round((e2 - e1) / 1e6 / 0.1, 2)
    except Exception:
        pass

    return None


def get_cpu_cores_used():
    try:
        return sum(1 for p in psutil.cpu_percent(percpu=True) if p > 1.0)
    except Exception:
        return None


# ============================================================
# 6. FLOPs helper
# ============================================================

def compute_model_flops(model, device, input_shape=(1, 1, 28, 28)):
    if not FVCORE_AVAILABLE:
        return None

    try:
        dummy = torch.ones(input_shape, dtype=torch.float32, device=device)
        fc = FlopCountAnalysis(model, dummy)
        fc.unsupported_ops_warnings(False)
        fc.uncalled_modules_warnings(False)
        return int(fc.total())
    except Exception as e:
        print(f"[FLOPs unavailable] {e}")
        return None


# ============================================================
# 7. Prediction quality helpers
# ============================================================

def get_prediction_quality(logits):
    probs = F.softmax(logits, dim=-1).squeeze()
    confidence = float(probs.max().item())

    top2 = torch.topk(logits.squeeze(), k=2).values
    margin = float((top2[0] - top2[1]).item())

    entropy = float(-(probs * torch.log(probs + 1e-12)).sum().item())

    return round(confidence, 6), round(margin, 6), round(entropy, 6)


# ============================================================
# 8. Energy tracking wrappers
# ============================================================

def _extract_energy_data(tracker, emissions_value):
    fd = getattr(tracker, "final_emissions_data", None)

    cpu_energy = getattr(fd, "cpu_energy", 0) if fd else 0
    gpu_energy = getattr(fd, "gpu_energy", 0) if fd else 0
    ram_energy = getattr(fd, "ram_energy", 0) if fd else 0
    total_energy = getattr(fd, "energy_consumed", 0) if fd else 0

    carbon_intensity = None
    if emissions_value and total_energy and total_energy > 0:
        carbon_intensity = round(emissions_value / total_energy, 8)

    return cpu_energy, gpu_energy, ram_energy, total_energy, carbon_intensity


def measure_agent_step(step_fn, *args, output_dir="./energy_logs", step_name="agent_step", measure_energy=True, **kwargs):
    """
    Measure any agent step, not only inference.

    Used for:
    [AGENT INIT]
    [AGENT TASK RECEIVED]
    [AGENT PLANNING]
    [AGENT ACTION]
    """

    os.makedirs(output_dir, exist_ok=True)

    if CODECARBON_AVAILABLE and measure_energy:
        tracker = EmissionsTracker(
            project_name=f"mnist_dnn_agent_{step_name}",
            output_dir=output_dir,
            output_file=f"codecarbon_dnn_{step_name}.csv",
            log_level="error",
            save_to_file=True,
        )

        tracker.start()

        t0 = time.perf_counter()
        result = step_fn(*args, **kwargs)
        exec_time = time.perf_counter() - t0

        emissions_value = tracker.stop()
        gpu_snap = get_gpu_metrics()

        cpu_energy, gpu_energy, ram_energy, total_energy, carbon_intensity = _extract_energy_data(
            tracker,
            emissions_value
        )

        return (
            result,
            exec_time,
            cpu_energy,
            gpu_energy,
            ram_energy,
            total_energy,
            emissions_value,
            carbon_intensity,
            gpu_snap,
        )

    t0 = time.perf_counter()
    result = step_fn(*args, **kwargs)
    exec_time = time.perf_counter() - t0
    gpu_snap = get_gpu_metrics()

    return result, exec_time, 0, 0, 0, 0, 0, None, gpu_snap


def run_with_energy_tracking(inference_fn, *args, output_dir="./energy_logs", **kwargs):
    """
    Backward-compatible wrapper for inference-only calls.
    It now uses measure_agent_step internally.
    """
    return measure_agent_step(
        inference_fn,
        *args,
        output_dir=output_dir,
        step_name="agent_action_inference",
        measure_energy=AGENT_STEP_CONFIG["[AGENT ACTION]"]["measure_energy"],
        **kwargs
    )


# ============================================================
# 9. Simple DNN Classifier
# ============================================================

class SimpleDNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.model(x)


# ============================================================
# 10. Train Simple DNN Classifier
# ============================================================

def train_dnn_classifier(model_path="simple_dnn_classifier.pth"):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    train_dataset = datasets.MNIST(
        root="./data",
        train=True,
        download=True,
        transform=transform
    )

    test_dataset = datasets.MNIST(
        root="./data",
        train=False,
        download=True,
        transform=transform
    )

    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=64,
        shuffle=True
    )

    test_loader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=1000,
        shuffle=False
    )

    device = DEVICE_EXEC
    print(f"Training device: {device}")

    classifier = SimpleDNNClassifier().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(classifier.parameters(), lr=0.001)

    for epoch in range(5):
        classifier.train()

        total_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = classifier(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(
            f"Epoch [{epoch + 1}/5] | "
            f"Loss: {total_loss / len(train_loader):.4f} | "
            f"Train Acc: {100 * correct / total:.2f}%"
        )

    classifier.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            _, predicted = torch.max(classifier(images), 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"\nInitial Test Accuracy: {100 * correct / total:.2f}%")

    torch.save(classifier.state_dict(), model_path)
    print(f"Saved: {model_path}")


def load_dnn_checkpoint_safely(model, classifier_path, device):
    state = torch.load(classifier_path, map_location=device)

    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]

    expected_keys = set(model.state_dict().keys())
    found_keys = set(state.keys()) if isinstance(state, dict) else set()

    if expected_keys != found_keys:
        missing = sorted(expected_keys - found_keys)
        unexpected = sorted(found_keys - expected_keys)
        raise RuntimeError(
            "DNN checkpoint key mismatch. "
            f"Missing keys: {missing}. Unexpected keys: {unexpected}."
        )

    model.load_state_dict(state)
    return model


# ============================================================
# 11. MNIST DNN Agent with full logging
# ============================================================

class MNISTDNNAgent:

    TRAINING_CONFIG = {
        "base_model":        "SimpleDNN-MNIST",
        "task":              "mnist_digit_classification",
        "dataset":           "torchvision.MNIST",
        "train_split":       "60000_samples",
        "test_split":        "10000_samples",
        "optimizer":         "Adam",
        "learning_rate":     0.001,
        "train_batch_size":  64,
        "eval_batch_size":   1,
        "num_train_epochs":  5,
        "weight_decay":      None,
        "input_shape":       "1x28x28",
        "num_labels":        10,
        "normalization_mean": 0.1307,
        "normalization_std":  0.3081,
        "best_model_metric": "test_accuracy",
        "loss_function":     "CrossEntropyLoss",
    }

    GENERATION_CONFIG = {
        "temperature":      "N/A",
        "top_p":            "N/A",
        "top_k":            "N/A",
        "do_sample":        False,
        "max_new_tokens":   "N/A",
        "decoding_used":    False,
        "inference_mode":   "image_classification",
        "prediction_rule":  "argmax_softmax",
    }

    def __init__(self, classifier_path="simple_dnn_classifier.pth"):
        self.classifier_name = "DNN"
        self.classifier_path = classifier_path
        self.device = DEVICE_EXEC

        self.classifier = SimpleDNNClassifier().to(self.device)
        self.classifier = load_dnn_checkpoint_safely(
            self.classifier,
            classifier_path,
            self.device
        )
        self.classifier.eval()

        self.test_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])

        self.model_param = sum(p.numel() for p in self.classifier.parameters())
        self.trainable_param = sum(
            p.numel() for p in self.classifier.parameters() if p.requires_grad
        )

        self.model_flops = compute_model_flops(self.classifier, self.device)

        self.model_dtype = str(next(self.classifier.parameters()).dtype)
        self.precision_type = "fp16" if "float16" in self.model_dtype else "fp32"
        self.fp16_enabled = "float16" in self.model_dtype
        self.fp32_enabled = "float32" in self.model_dtype

        self.arch_meta = {
            "hidden_sizes":       "784 -> 128 -> 64 -> 10",
            "num_hidden_layers":  3,
            "activation":         "ReLU",
            "context_window_size": "N/A",
            "vocab_size":          "N/A",
            "pad_token_id":        "N/A",
        }

        if self.model_flops:
            print(f"Model FLOPs (28x28 input): {self.model_flops:,}")
        else:
            print("FLOPs unavailable.")

    # ------------------------------------------------------------
    # Agent step payloads
    # ------------------------------------------------------------

    def _agent_init_step(self):
        return (
            f"Agent initialized on device {self.device}; "
            f"classifier loaded: {self.classifier_name}; "
            f"params: {self.model_param:,}; "
            f"FLOPs: {self.model_flops}"
        )

    def _agent_task_received_step(self, task):
        return f"Task received from user: {task}"

    def _agent_planning_step(self, task):
        task_lower = task.lower()

        if "test" in task_lower or "classify" in task_lower:
            return {
                "task_supported": True,
                "message": "Task supported. Plan: load MNIST test set and classify each image using DNN."
            }

        return {
            "task_supported": False,
            "message": "Task not supported. This agent only supports MNIST classification."
        }

    # ------------------------------------------------------------
    # Inference
    # ------------------------------------------------------------

    def _infer(self, image_tensor):
        with torch.no_grad():
            logits = self.classifier(image_tensor)
            probs = torch.softmax(logits, dim=1)
            confidence, predicted = torch.max(probs, dim=1)

        return logits, int(predicted.item()), float(confidence.item())

    # ------------------------------------------------------------
    # Log row builder
    # ------------------------------------------------------------

    def _build_log_row(
        self,
        task_id,
        agent_step,
        message,
        step_time,
        true_label=None,
        predicted_label=None,
        logits=None,
        cpu_energy=0,
        gpu_energy=0,
        ram_energy=0,
        total_energy=0,
        emissions_value=0,
        carbon_intensity=None,
        gpu_metrics=None,
        model_accuracy=None,
        model_precision_weighted=None,
        model_recall_weighted=None,
        model_f1_weighted=None,
    ):
        gpu_metrics = gpu_metrics or {}

        correct = None
        confidence_score = None
        logit_margin = None
        entropy = None

        if true_label is not None and predicted_label is not None:
            correct = int(predicted_label) == int(true_label)

        if logits is not None:
            confidence_score, logit_margin, entropy = get_prediction_quality(logits)

        # DNN/CNN image-style token proxy
        input_tokens = 784
        output_tokens = 1
        total_tokens = input_tokens + output_tokens

        tokens_per_second = round(total_tokens / step_time, 4) if step_time > 0 else None

        total_energy = total_energy if total_energy is not None else 0.0
        cpu_energy = cpu_energy if cpu_energy is not None else 0.0
        gpu_energy = gpu_energy if gpu_energy is not None else 0.0
        ram_energy = ram_energy if ram_energy is not None else 0.0

        joules_per_token = 0.0
        energy_per_token_kwh = 0.0
        watts_estimated = 0.0
        gpu_energy_pct = 0.0
        cpu_energy_pct = 0.0

        if total_energy > 0 and total_tokens > 0:
            energy_per_token_kwh = round(total_energy / total_tokens, 12)
            joules_per_token = round((total_energy * 3_600_000) / total_tokens, 6)

            if step_time > 0:
                watts_estimated = round((total_energy * 3_600_000) / step_time, 4)

            gpu_energy_pct = round((gpu_energy / total_energy) * 100, 2)
            cpu_energy_pct = round((cpu_energy / total_energy) * 100, 2)

        cpu_usage = get_cpu_usage()
        ram_usage = get_ram_usage()
        cpu_freq = get_cpu_freq()
        cpu_temp = get_cpu_temp()
        cpu_power_w = get_cpu_power_draw_w()
        cpu_cores_used = get_cpu_cores_used()

        tc = self.TRAINING_CONFIG
        gc = self.GENERATION_CONFIG
        step_cfg = AGENT_STEP_CONFIG.get(agent_step, {})

        return {
            # --- Identity ---
            "timestamp":           time.strftime("%Y-%m-%d %H:%M:%S"),
            "unique_device_id":    DEVICE_UUID,
            "device_short_id":     DEVICE_SHORT,
            "pc_name":             get_hostname(),
            "collection_mode":     "automated",

            # --- Agent step ---
            "task_id":             task_id,
            "sample_index":        task_id - 1 if task_id > 0 else None,
            "agent_step":          agent_step,
            "agent_step_description": step_cfg.get("description"),
            "agent_step_energy_enabled": step_cfg.get("measure_energy"),
            "classifier":          self.classifier_name,
            "message":             message,
            "step_time_seconds":   format(step_time, ".10f"),
            "execution_time_sec":  format(step_time, ".10f"),

            # --- Prediction ---
            "true_label":          true_label,
            "predicted_label":     predicted_label,
            "prediction":          predicted_label,
            "correct":             correct,
            "confidence_score":    confidence_score,
            "logit_margin":        logit_margin,
            "entropy":             entropy,

            # --- Model identity ---
            "model_type":          "SimpleDNN-MNIST",
            "base_model":          tc["base_model"],
            "task":                tc["task"],
            "dataset":             tc["dataset"],

            # --- Parameters ---
            "parameters":          self.model_param,
            "trainable_parameters": self.trainable_param,
            "model_param":         self.model_param,
            "model_flops":         self.model_flops,

            # --- CodeCarbon energy ---
            "cpu_energy_kwh":             cpu_energy,
            "gpu_energy_kwh":             gpu_energy,
            "ram_energy_kwh":             ram_energy,
            "total_energy_kwh":           total_energy,
            "total_emissions_kg":         emissions_value,
            "carbon_intensity_kgco2_kwh": carbon_intensity,

            # --- Efficiency derived ---
            "input_tokens":               input_tokens,
            "output_tokens":              output_tokens,
            "total_tokens":               total_tokens,
            "tokens_per_second":          tokens_per_second,
            "joules_per_token":           joules_per_token,
            "energy_per_token_kwh":       energy_per_token_kwh,
            "watts_estimated":            watts_estimated,
            "gpu_energy_pct_of_total":    gpu_energy_pct,
            "cpu_energy_pct_of_total":    cpu_energy_pct,
            "batch_size_at_inference":    1,

            # --- CPU hardware ---
            "cpu_model":           CPU_MODEL_NAME,
            "cpu_architecture":    CPU_ARCH,
            "cpu_core_count":      CPU_CORE_COUNT,
            "cpu_thread_count":    CPU_THREAD_COUNT,
            "cpu_tdp_w":           CPU_TDP_W,
            "cpu_usage_pct":       cpu_usage,
            "cpu_clock_mhz":       cpu_freq,
            "cpu_temp_c":          cpu_temp,
            "cpu_power_draw_w":    cpu_power_w,
            "cpu_cores_used":      cpu_cores_used,

            # --- GPU hardware ---
            "gpu_model":                get_gpu_name(),
            "gpu_driver_version":       GPU_STATIC["gpu_driver_version"],
            "gpu_compute_capability":   GPU_STATIC["gpu_compute_capability"],
            "gpu_power_limit_w":        GPU_STATIC["gpu_power_limit_w"],
            "gpu_memory_total_mb":      GPU_STATIC["gpu_memory_total_mb"],
            "gpu_power_draw_w":         gpu_metrics.get("gpu_power_draw_w"),
            "gpu_utilization_pct":      gpu_metrics.get("gpu_utilization_pct"),
            "gpu_temp_c":               gpu_metrics.get("gpu_temp_c"),
            "gpu_memory_used_mb":       gpu_metrics.get("gpu_memory_used_mb"),
            "gpu_sm_clock_mhz":         gpu_metrics.get("gpu_sm_clock_mhz"),
            "gpu_memory_clock_mhz":     gpu_metrics.get("gpu_memory_clock_mhz"),
            "cuda_driver_version":      CUDA_DRIVER_VERSION,
            "cuda_available":           torch.cuda.is_available(),
            "device_type":              str(self.device),

            # --- RAM / memory ---
            "ram_usage_pct":        ram_usage,
            "memory_footprint_mb":  get_memory_footprint_mb(),
            "system_ram_total_gb":  SYSTEM_RAM_TOTAL_GB,

            # --- Environment ---
            "os_name":              OS_NAME,
            "os_version":           OS_VERSION,
            "os_architecture":      OS_ARCHITECTURE,
            "os_full_name":         OS_FULL_NAME,
            "python_version":       PYTHON_VERSION,
            "torch_version":        TORCH_VERSION,
            "codecarbon_version":   CODECARBON_VERSION,

            # --- Precision ---
            "model_dtype":          self.model_dtype,
            "precision_type":       self.precision_type,
            "fp16_enabled":         self.fp16_enabled,
            "fp32_enabled":         self.fp32_enabled,

            # --- Architecture ---
            "hidden_sizes":         self.arch_meta["hidden_sizes"],
            "num_hidden_layers":    self.arch_meta["num_hidden_layers"],
            "activation":           self.arch_meta["activation"],
            "context_window_size":  self.arch_meta["context_window_size"],
            "vocab_size":           self.arch_meta["vocab_size"],
            "pad_token_id":         self.arch_meta["pad_token_id"],

            # --- Generation / inference config ---
            "temperature":          gc["temperature"],
            "top_p":                gc["top_p"],
            "top_k":                gc["top_k"],
            "do_sample":            gc["do_sample"],
            "max_new_tokens":       gc["max_new_tokens"],
            "decoding_used":        gc["decoding_used"],
            "inference_mode":       gc["inference_mode"],
            "prediction_rule":      gc["prediction_rule"],

            # --- Training config ---
            "optimizer":            tc["optimizer"],
            "learning_rate":        tc["learning_rate"],
            "train_batch_size":     tc["train_batch_size"],
            "eval_batch_size":      tc["eval_batch_size"],
            "num_train_epochs":     tc["num_train_epochs"],
            "weight_decay":         tc["weight_decay"],
            "input_shape":          tc["input_shape"],
            "num_labels":           tc["num_labels"],
            "normalization_mean":   tc["normalization_mean"],
            "normalization_std":    tc["normalization_std"],
            "best_model_metric":    tc["best_model_metric"],
            "loss_function":        tc["loss_function"],
            "train_split":          tc["train_split"],
            "test_split":           tc["test_split"],

            # --- Final model metrics ---
            "model_accuracy":           model_accuracy,
            "model_precision_weighted": model_precision_weighted,
            "model_recall_weighted":    model_recall_weighted,
            "model_f1_weighted":        model_f1_weighted,
        }

    def _print_log(self, row):
        print(
            f"Task {row['task_id']} | "
            f"{row['agent_step']} | "
            f"Classifier: {row['classifier']} | "
            f"{row['message']} | "
            f"Time: {row['step_time_seconds']} sec | "
            f"Energy: {row['total_energy_kwh']} kWh | "
            f"Watts: {row['watts_estimated']}"
        )

    def _save_logs(self, logs, log_file):
        df = pd.DataFrame(logs)

        if SAVE_CSV:
            df.to_csv(log_file, index=False)
            print(f"CSV saved to: {log_file}")

        if SAVE_EXCEL:
            xlsx_file = log_file.replace(".csv", ".xlsx")
            df.to_excel(xlsx_file, index=False)
            print(f"Excel saved to: {xlsx_file}")

        return df

    def run(self, task, log_file="mnist_agent_dnn_full_log.csv"):
        all_logs = []

        # =============================================
        # Step 1: [AGENT INIT]
        # =============================================
        (
            msg,
            step_time,
            cpu_energy,
            gpu_energy,
            ram_energy,
            total_energy,
            emissions_value,
            carbon_intensity,
            gpu_snap,
        ) = measure_agent_step(
            self._agent_init_step,
            step_name="agent_init",
            measure_energy=AGENT_STEP_CONFIG["[AGENT INIT]"]["measure_energy"]
        )

        row = self._build_log_row(
            task_id=0,
            agent_step="[AGENT INIT]",
            message=msg,
            step_time=step_time,
            cpu_energy=cpu_energy,
            gpu_energy=gpu_energy,
            ram_energy=ram_energy,
            total_energy=total_energy,
            emissions_value=emissions_value,
            carbon_intensity=carbon_intensity,
            gpu_metrics=gpu_snap,
        )

        all_logs.append(row)
        self._print_log(row)

        # =============================================
        # Step 2: [AGENT TASK RECEIVED]
        # =============================================
        (
            msg,
            step_time,
            cpu_energy,
            gpu_energy,
            ram_energy,
            total_energy,
            emissions_value,
            carbon_intensity,
            gpu_snap,
        ) = measure_agent_step(
            self._agent_task_received_step,
            task,
            step_name="agent_task_received",
            measure_energy=AGENT_STEP_CONFIG["[AGENT TASK RECEIVED]"]["measure_energy"]
        )

        row = self._build_log_row(
            task_id=0,
            agent_step="[AGENT TASK RECEIVED]",
            message=msg,
            step_time=step_time,
            cpu_energy=cpu_energy,
            gpu_energy=gpu_energy,
            ram_energy=ram_energy,
            total_energy=total_energy,
            emissions_value=emissions_value,
            carbon_intensity=carbon_intensity,
            gpu_metrics=gpu_snap,
        )

        all_logs.append(row)
        self._print_log(row)

        # =============================================
        # Step 3: [AGENT PLANNING]
        # =============================================
        (
            planning_result,
            step_time,
            cpu_energy,
            gpu_energy,
            ram_energy,
            total_energy,
            emissions_value,
            carbon_intensity,
            gpu_snap,
        ) = measure_agent_step(
            self._agent_planning_step,
            task,
            step_name="agent_planning",
            measure_energy=AGENT_STEP_CONFIG["[AGENT PLANNING]"]["measure_energy"]
        )

        task_supported = planning_result["task_supported"]
        msg = planning_result["message"]

        row = self._build_log_row(
            task_id=0,
            agent_step="[AGENT PLANNING]",
            message=msg,
            step_time=step_time,
            cpu_energy=cpu_energy,
            gpu_energy=gpu_energy,
            ram_energy=ram_energy,
            total_energy=total_energy,
            emissions_value=emissions_value,
            carbon_intensity=carbon_intensity,
            gpu_metrics=gpu_snap,
        )

        all_logs.append(row)
        self._print_log(row)

        if not task_supported:
            self._save_logs(all_logs, log_file)
            return {
                "status": "failed",
                "message": msg,
                "log_file": log_file,
            }

        # =============================================
        # Step 4: [AGENT ACTION]
        # =============================================
        return self._run_action_loop(all_logs, log_file)

    def _run_action_loop(self, all_logs, log_file):
        print("\n[AGENT ACTION]")
        print("Loading MNIST test dataset and classifying images...\n")

        test_dataset = datasets.MNIST(
            root="./data",
            train=False,
            download=True,
            transform=self.test_transform
        )

        if NUM_TEST_SAMPLES is None:
            total_tasks = len(test_dataset)
        else:
            total_tasks = min(int(NUM_TEST_SAMPLES), len(test_dataset))

        print(f"Configured NUM_TEST_SAMPLES: {NUM_TEST_SAMPLES}")
        print(f"Total samples to classify: {total_tasks}")

        correct_count = 0
        wrong_count = 0
        total_action_time = 0.0

        class_correct = [0] * 10
        class_total = [0] * 10

        true_ids = []
        pred_ids = []

        overall_start = time.perf_counter()

        for task_index in range(total_tasks):
            task_id = task_index + 1

            image, true_label = test_dataset[task_index]
            true_label = int(true_label)

            image_tensor = image.unsqueeze(0).to(self.device)

            (
                (logits, predicted_label, confidence_value),
                exec_time,
                cpu_energy,
                gpu_energy,
                ram_energy,
                total_energy,
                emissions_value,
                carbon_intensity,
                gpu_snap,
            ) = run_with_energy_tracking(
                self._infer,
                image_tensor,
                output_dir="./energy_logs"
            )

            total_action_time += exec_time

            correct = predicted_label == true_label

            if correct:
                correct_count += 1
                class_correct[true_label] += 1
            else:
                wrong_count += 1

            class_total[true_label] += 1

            true_ids.append(true_label)
            pred_ids.append(predicted_label)

            msg = (
                f"Classified MNIST image. "
                f"True: {true_label}, Predicted: {predicted_label}, "
                f"Confidence: {confidence_value:.6f}, Correct: {correct}"
            )

            row = self._build_log_row(
                task_id=task_id,
                agent_step="[AGENT ACTION]",
                message=msg,
                step_time=exec_time,
                true_label=true_label,
                predicted_label=predicted_label,
                logits=logits,
                cpu_energy=cpu_energy,
                gpu_energy=gpu_energy,
                ram_energy=ram_energy,
                total_energy=total_energy,
                emissions_value=emissions_value,
                carbon_intensity=carbon_intensity,
                gpu_metrics=gpu_snap,
            )

            all_logs.append(row)
            self._print_log(row)

            if (task_index + 1) % 500 == 0 or (task_index + 1) == total_tasks:
                print(f"  Completed {task_index + 1}/{total_tasks} samples")

        overall_runtime = time.perf_counter() - overall_start

        if SKLEARN_AVAILABLE:
            accuracy = accuracy_score(true_ids, pred_ids)
            prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
                true_ids,
                pred_ids,
                average="weighted",
                zero_division=0
            )
        else:
            accuracy = correct_count / total_tasks
            prec_w = None
            rec_w = None
            f1_w = None

        for row in all_logs:
            row["model_accuracy"] = accuracy
            row["model_precision_weighted"] = prec_w
            row["model_recall_weighted"] = rec_w
            row["model_f1_weighted"] = f1_w

        self._save_logs(all_logs, log_file)

        per_class_accuracy = {
            d: round(100 * class_correct[d] / class_total[d], 2)
            if class_total[d] > 0 else 0.0
            for d in range(10)
        }

        print("\n" + "=" * 60)
        print("Inference logging complete")
        print("=" * 60)
        print(f"Rows   : {len(all_logs)}")
        print(f"Columns: {len(all_logs[0]) if all_logs else 0}")
        print("\nFinal metrics:")
        print(f"  Accuracy           : {accuracy:.4f}")

        if prec_w is not None:
            print(f"  Weighted Precision : {prec_w:.4f}")
            print(f"  Weighted Recall    : {rec_w:.4f}")
            print(f"  Weighted F1        : {f1_w:.4f}")

        return {
            "status": "success",
            "task": "MNIST classification with full DNN agent log",
            "classifier": self.classifier_name,
            "total_images_classified": total_tasks,
            "correct_predictions": correct_count,
            "wrong_predictions": wrong_count,
            "test_accuracy_percent": round(accuracy * 100, 2),
            "weighted_precision": round(prec_w, 4) if prec_w is not None else None,
            "weighted_recall": round(rec_w, 4) if rec_w is not None else None,
            "weighted_f1": round(f1_w, 4) if f1_w is not None else None,
            "total_action_time_seconds": format(total_action_time, ".10f"),
            "average_action_time_per_image_seconds": format(
                total_action_time / total_tasks,
                ".10f"
            ),
            "overall_runtime_seconds": format(overall_runtime, ".10f"),
            "log_file": log_file.replace(".csv", ".xlsx") if SAVE_EXCEL else log_file,
            "per_class_accuracy_percent": per_class_accuracy,
        }


# ============================================================
# 12. Main
# ============================================================

if __name__ == "__main__":
    classifier_path = "simple_dnn_classifier.pth"

    if not os.path.exists(classifier_path):
        print("No saved DNN classifier found. Training a new DNN classifier.")
        train_dnn_classifier(model_path=classifier_path)
    else:
        print("Saved Simple DNN classifier found. Checking compatibility.")

    try:
        agent = MNISTDNNAgent(classifier_path=classifier_path)
    except RuntimeError:
        print("\nExisting DNN checkpoint is incompatible with this DNN class.")
        print("Deleting old checkpoint and retraining a fresh DNN classifier.\n")

        try:
            os.remove(classifier_path)
        except Exception as e:
            print(f"Could not delete old checkpoint: {e}")

        train_dnn_classifier(model_path=classifier_path)
        agent = MNISTDNNAgent(classifier_path=classifier_path)

    output = agent.run(
        task="Classify all 10000 MNIST test images using Simple DNN",
        log_file="mnist_agent_dnn_full_log.csv",
    )

    print("\n[AGENT FINAL OUTPUT]")
    print(output)

    try:
        if NVML_AVAILABLE:
            pynvml.nvmlShutdown()
    except Exception:
        pass


Saved Simple DNN classifier found. Checking compatibility.
Model FLOPs (28x28 input): 109,184
Task 0 | [AGENT INIT] | Classifier: DNN | Agent initialized on device cuda; classifier loaded: DNN; params: 109,386; FLOPs: 109184 | Time: 0.0000183000 sec | Energy: 1.063335025941746e-06 kWh | Watts: 209180.3987
Task 0 | [AGENT TASK RECEIVED] | Classifier: DNN | Task received from user: Classify all 10000 MNIST test images using Simple DNN | Time: 0.0000021001 sec | Energy: 7.028772405141758e-07 kWh | Watts: 1204856.4457
Task 0 | [AGENT PLANNING] | Classifier: DNN | Task supported. Plan: load MNIST test set and classify each image using DNN. | Time: 0.0000038999 sec | Energy: 1.2335238926526573e-06 kWh | Watts: 1138662.7582

[AGENT ACTION]
Loading MNIST test dataset and classifying images...

Configured NUM_TEST_SAMPLES: 10
Total samples to classify: 10
Task 1 | [AGENT ACTION] | Classifier: DNN | Classified MNIST image. True: 7, Predicted: 7, Confidence: 0.994217, Correct: True | Time: 0.0043

# CNN

In [7]:
import os
import sys
import time
import socket
import platform
import hashlib

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import psutil
from torchvision import datasets, transforms


# ============================================================
# 0. RUN CONFIGURATION
# ============================================================

# Device configuration:
# "auto" = CUDA if available, otherwise CPU
# "cpu"  = force CPU
# "cuda" = force CUDA if available
DEVICE_MODE = "auto"

# Number of MNIST test samples to run.
# Use None for all 10,000 samples.
# Use small number like 10 for debugging.
NUM_TEST_SAMPLES = 10

# Save outputs
SAVE_EXCEL = True
SAVE_CSV = False

# Measure energy for all four agent steps.
# Warning: very short steps may have noisy CodeCarbon values.
AGENT_STEP_CONFIG = {
    "[AGENT INIT]": {
        "measure_energy": True,
        "description": "Initialize the AI agent and load model metadata."
    },
    "[AGENT TASK RECEIVED]": {
        "measure_energy": True,
        "description": "Receive the user task."
    },
    "[AGENT PLANNING]": {
        "measure_energy": True,
        "description": "Check task type and build the execution plan."
    },
    "[AGENT ACTION]": {
        "measure_energy": True,
        "description": "Run per-sample MNIST classification."
    },
}


def resolve_device():
    if DEVICE_MODE.lower() == "cpu":
        return torch.device("cpu")

    if DEVICE_MODE.lower() == "cuda":
        if torch.cuda.is_available():
            return torch.device("cuda")
        print("CUDA requested but not available. Falling back to CPU.")
        return torch.device("cpu")

    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE_EXEC = resolve_device()


# ============================================================
# 1. Safe optional imports
# ============================================================

# ---- CodeCarbon ----
try:
    from codecarbon import EmissionsTracker
    import codecarbon
    CODECARBON_AVAILABLE = True
    CODECARBON_VERSION = codecarbon.__version__
except Exception:
    EmissionsTracker = None
    CODECARBON_AVAILABLE = False
    CODECARBON_VERSION = "unavailable"
    print("CodeCarbon not available. Energy values will be set to 0.")

# ---- pynvml ----
try:
    import pynvml
    pynvml.nvmlInit()
    NVML_AVAILABLE = True
    NVML_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0) if torch.cuda.is_available() else None
except Exception:
    NVML_AVAILABLE = False
    NVML_HANDLE = None

# ---- py-cpuinfo ----
try:
    import cpuinfo
    _CPU_INFO = cpuinfo.get_cpu_info()
    CPU_MODEL_NAME = _CPU_INFO.get("brand_raw", "Unknown")
    CPU_ARCH = _CPU_INFO.get("arch", platform.machine())
    CPU_TDP_W = None
except Exception:
    CPU_MODEL_NAME = "Unknown"
    CPU_ARCH = platform.machine()
    CPU_TDP_W = None

# ---- fvcore: FLOPs ----
try:
    from fvcore.nn import FlopCountAnalysis
    FVCORE_AVAILABLE = True
except Exception:
    FlopCountAnalysis = None
    FVCORE_AVAILABLE = False

# ---- sklearn metrics ----
try:
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    SKLEARN_AVAILABLE = True
except Exception:
    accuracy_score = None
    precision_recall_fscore_support = None
    SKLEARN_AVAILABLE = False
    print("sklearn not available. Final precision/recall/F1 will be set to None.")


# ============================================================
# 2. OS / environment helpers
# ============================================================

def get_os_full_name():
    system = platform.system()
    architecture = platform.machine()

    if system == "Windows":
        try:
            import winreg
            key = winreg.OpenKey(
                winreg.HKEY_LOCAL_MACHINE,
                r"SOFTWARE\Microsoft\Windows NT\CurrentVersion"
            )
            product_name = winreg.QueryValueEx(key, "ProductName")[0]
            display_version = winreg.QueryValueEx(key, "DisplayVersion")[0]
            current_build = winreg.QueryValueEx(key, "CurrentBuild")[0]
            return f"{product_name} {display_version} Build {current_build} {architecture}"
        except Exception:
            return f"Windows {platform.release()} {architecture}"

    if system == "Linux":
        os_info = {}
        try:
            with open("/etc/os-release", "r") as file:
                for line in file:
                    if "=" in line:
                        key, value = line.strip().split("=", 1)
                        os_info[key] = value.strip('"')
        except Exception:
            pass

        pretty_name = os_info.get("PRETTY_NAME")
        name = os_info.get("NAME")
        version = os_info.get("VERSION")
        version_id = os_info.get("VERSION_ID")
        distro_id = os_info.get("ID")

        if pretty_name:
            return f"{pretty_name} {architecture}"
        if name and version:
            return f"{name} {version} {architecture}"
        if name and version_id:
            return f"{name} {version_id} {architecture}"
        if distro_id:
            return f"{distro_id} {platform.release()} {architecture}"
        return f"Linux {platform.release()} {architecture}"

    if system == "Darwin":
        return f"macOS {platform.mac_ver()[0]} {architecture}"

    return f"{system} {platform.release()} {architecture}"


TORCH_VERSION = torch.__version__
PYTHON_VERSION = sys.version.split()[0]
OS_NAME = platform.system()
OS_VERSION = platform.version()
OS_ARCHITECTURE = platform.machine()
OS_FULL_NAME = get_os_full_name()
SYSTEM_RAM_TOTAL_GB = round(psutil.virtual_memory().total / (1024 ** 3), 2)
CPU_CORE_COUNT = psutil.cpu_count(logical=False)
CPU_THREAD_COUNT = psutil.cpu_count(logical=True)


# ============================================================
# 3. Stable device ID
# ============================================================

def make_stable_device_id():
    raw = f"{socket.gethostname()}-{platform.system()}-{platform.machine()}-{CPU_MODEL_NAME}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


DEVICE_UUID = make_stable_device_id()
DEVICE_SHORT = DEVICE_UUID[:8]


# ============================================================
# 4. GPU static info
# ============================================================

def _get_cuda_driver_version():
    if not NVML_AVAILABLE:
        return None
    try:
        driver = pynvml.nvmlSystemGetDriverVersion()
        return driver.decode("utf-8") if isinstance(driver, bytes) else driver
    except Exception:
        return None


CUDA_DRIVER_VERSION = _get_cuda_driver_version()


def _get_gpu_static():
    defaults = {
        "gpu_power_limit_w": None,
        "gpu_driver_version": CUDA_DRIVER_VERSION,
        "gpu_memory_total_mb": None,
        "gpu_compute_capability": None,
    }

    if not NVML_AVAILABLE or NVML_HANDLE is None:
        return defaults

    try:
        power_limit_mw = pynvml.nvmlDeviceGetPowerManagementLimit(NVML_HANDLE)
        mem_info = pynvml.nvmlDeviceGetMemoryInfo(NVML_HANDLE)
        cc_major, cc_minor = pynvml.nvmlDeviceGetCudaComputeCapability(NVML_HANDLE)

        return {
            "gpu_power_limit_w": round(power_limit_mw / 1000.0, 1),
            "gpu_driver_version": CUDA_DRIVER_VERSION,
            "gpu_memory_total_mb": round(mem_info.total / (1024 ** 2), 2),
            "gpu_compute_capability": f"{cc_major}.{cc_minor}",
        }
    except Exception:
        return defaults


GPU_STATIC = _get_gpu_static()


# ============================================================
# 5. Per-sample device helpers
# ============================================================

def get_hostname():
    return socket.gethostname()


def get_gpu_name():
    return torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU"


def get_cpu_usage():
    return psutil.cpu_percent(interval=None)


def get_ram_usage():
    return psutil.virtual_memory().percent


def get_cpu_freq():
    freq = psutil.cpu_freq()
    return round(freq.current, 2) if freq else None


def get_memory_footprint_mb():
    process = psutil.Process(os.getpid())
    return round(process.memory_info().rss / (1024 * 1024), 4)


def get_gpu_metrics():
    null = {
        "gpu_power_draw_w": None,
        "gpu_utilization_pct": None,
        "gpu_temp_c": None,
        "gpu_memory_used_mb": None,
        "gpu_sm_clock_mhz": None,
        "gpu_memory_clock_mhz": None,
    }

    if not NVML_AVAILABLE or NVML_HANDLE is None:
        return null

    try:
        power_mw = pynvml.nvmlDeviceGetPowerUsage(NVML_HANDLE)
        util = pynvml.nvmlDeviceGetUtilizationRates(NVML_HANDLE)
        temp = pynvml.nvmlDeviceGetTemperature(NVML_HANDLE, pynvml.NVML_TEMPERATURE_GPU)
        mem_info = pynvml.nvmlDeviceGetMemoryInfo(NVML_HANDLE)
        sm_clock = pynvml.nvmlDeviceGetClockInfo(NVML_HANDLE, pynvml.NVML_CLOCK_SM)
        mem_clock = pynvml.nvmlDeviceGetClockInfo(NVML_HANDLE, pynvml.NVML_CLOCK_MEM)

        return {
            "gpu_power_draw_w": round(power_mw / 1000.0, 2),
            "gpu_utilization_pct": util.gpu,
            "gpu_temp_c": temp,
            "gpu_memory_used_mb": round(mem_info.used / (1024 ** 2), 2),
            "gpu_sm_clock_mhz": sm_clock,
            "gpu_memory_clock_mhz": mem_clock,
        }
    except Exception:
        return null


def get_cpu_temp():
    try:
        temps = psutil.sensors_temperatures()
        if not temps:
            return None

        for key in ("coretemp", "k10temp", "cpu_thermal", "acpitz"):
            if key in temps:
                values = [e.current for e in temps[key] if e.current and e.current > 0]
                if values:
                    return round(sum(values) / len(values), 1)
    except Exception:
        pass

    return None


def get_cpu_power_draw_w():
    rapl_path = "/sys/class/powercap/intel-rapl/intel-rapl:0/energy_uj"

    try:
        if os.path.exists(rapl_path):
            with open(rapl_path) as f:
                e1 = int(f.read().strip())

            time.sleep(0.1)

            with open(rapl_path) as f:
                e2 = int(f.read().strip())

            return round((e2 - e1) / 1e6 / 0.1, 2)
    except Exception:
        pass

    return None


def get_cpu_cores_used():
    try:
        return sum(1 for p in psutil.cpu_percent(percpu=True) if p > 1.0)
    except Exception:
        return None


# ============================================================
# 6. FLOPs helper
# ============================================================

def compute_model_flops(model, device, input_shape=(1, 1, 28, 28)):
    if not FVCORE_AVAILABLE:
        return None

    try:
        dummy = torch.ones(input_shape, dtype=torch.float32, device=device)
        fc = FlopCountAnalysis(model, dummy)
        fc.unsupported_ops_warnings(False)
        fc.uncalled_modules_warnings(False)
        return int(fc.total())
    except Exception as e:
        print(f"[FLOPs unavailable] {e}")
        return None


# ============================================================
# 7. Prediction quality helpers
# ============================================================

def get_prediction_quality(logits):
    probs = F.softmax(logits, dim=-1).squeeze()
    confidence = float(probs.max().item())

    top2 = torch.topk(logits.squeeze(), k=2).values
    margin = float((top2[0] - top2[1]).item())

    entropy = float(-(probs * torch.log(probs + 1e-12)).sum().item())

    return round(confidence, 6), round(margin, 6), round(entropy, 6)


# ============================================================
# 8. Energy tracking wrappers
# ============================================================

def _extract_energy_data(tracker, emissions_value):
    fd = getattr(tracker, "final_emissions_data", None)

    cpu_energy = getattr(fd, "cpu_energy", 0) if fd else 0
    gpu_energy = getattr(fd, "gpu_energy", 0) if fd else 0
    ram_energy = getattr(fd, "ram_energy", 0) if fd else 0
    total_energy = getattr(fd, "energy_consumed", 0) if fd else 0

    carbon_intensity = None
    if emissions_value and total_energy and total_energy > 0:
        carbon_intensity = round(emissions_value / total_energy, 8)

    return cpu_energy, gpu_energy, ram_energy, total_energy, carbon_intensity


def measure_agent_step(step_fn, *args, output_dir="./energy_logs", step_name="agent_step", measure_energy=True, **kwargs):
    """
    Measure any agent step, not only inference.

    Used for:
    [AGENT INIT]
    [AGENT TASK RECEIVED]
    [AGENT PLANNING]
    [AGENT ACTION]
    """

    os.makedirs(output_dir, exist_ok=True)

    if CODECARBON_AVAILABLE and measure_energy:
        tracker = EmissionsTracker(
            project_name=f"mnist_agent_{step_name}",
            output_dir=output_dir,
            output_file=f"codecarbon_{step_name}.csv",
            log_level="error",
            save_to_file=True,
        )

        tracker.start()

        t0 = time.perf_counter()
        result = step_fn(*args, **kwargs)
        exec_time = time.perf_counter() - t0

        emissions_value = tracker.stop()
        gpu_snap = get_gpu_metrics()

        cpu_energy, gpu_energy, ram_energy, total_energy, carbon_intensity = _extract_energy_data(
            tracker,
            emissions_value
        )

        return (
            result,
            exec_time,
            cpu_energy,
            gpu_energy,
            ram_energy,
            total_energy,
            emissions_value,
            carbon_intensity,
            gpu_snap,
        )

    t0 = time.perf_counter()
    result = step_fn(*args, **kwargs)
    exec_time = time.perf_counter() - t0
    gpu_snap = get_gpu_metrics()

    return result, exec_time, 0, 0, 0, 0, 0, None, gpu_snap


def run_with_energy_tracking(inference_fn, *args, output_dir="./energy_logs", **kwargs):
    """
    Backward-compatible wrapper for inference-only calls.
    It now uses measure_agent_step internally.
    """
    return measure_agent_step(
        inference_fn,
        *args,
        output_dir=output_dir,
        step_name="agent_action_inference",
        measure_energy=AGENT_STEP_CONFIG["[AGENT ACTION]"]["measure_energy"],
        **kwargs
    )


# ============================================================
# 9. Simple CNN Classifier
# ============================================================

class SimpleCNNClassifier(nn.Module):
    """
    CNN architecture with explicit layer names:
    conv1, conv2, fc1, fc2

    This matches checkpoints that contain:
    conv1.weight, conv1.bias,
    conv2.weight, conv2.bias,
    fc1.weight, fc1.bias,
    fc2.weight, fc2.bias
    """

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # 1x28x28 -> 16x14x14
        x = self.pool(F.relu(self.conv2(x)))   # 16x14x14 -> 32x7x7

        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x


# ============================================================
# 10. Train Simple CNN Classifier
# ============================================================

def train_cnn_classifier(model_path="simple_cnn_classifier.pth"):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])

    train_dataset = datasets.MNIST(
        root="./data",
        train=True,
        download=True,
        transform=transform
    )

    test_dataset = datasets.MNIST(
        root="./data",
        train=False,
        download=True,
        transform=transform
    )

    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=64,
        shuffle=True
    )

    test_loader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=1000,
        shuffle=False
    )

    device = DEVICE_EXEC
    print(f"Training device: {device}")

    classifier = SimpleCNNClassifier().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(classifier.parameters(), lr=0.001)

    for epoch in range(5):
        classifier.train()

        total_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = classifier(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(
            f"Epoch [{epoch + 1}/5] | "
            f"Loss: {total_loss / len(train_loader):.4f} | "
            f"Train Acc: {100 * correct / total:.2f}%"
        )

    classifier.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            _, predicted = torch.max(classifier(images), 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f"\nInitial Test Accuracy: {100 * correct / total:.2f}%")

    torch.save(classifier.state_dict(), model_path)
    print(f"Saved: {model_path}")


def load_cnn_checkpoint_safely(model, classifier_path, device):
    """
    Load CNN checkpoint safely.

    If an incompatible checkpoint is found, this function raises RuntimeError.
    Main block catches it and retrains the CNN checkpoint.
    """
    state = torch.load(classifier_path, map_location=device)

    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]

    expected_keys = set(model.state_dict().keys())
    found_keys = set(state.keys()) if isinstance(state, dict) else set()

    if expected_keys != found_keys:
        missing = sorted(expected_keys - found_keys)
        unexpected = sorted(found_keys - expected_keys)
        raise RuntimeError(
            "CNN checkpoint key mismatch. "
            f"Missing keys: {missing}. Unexpected keys: {unexpected}."
        )

    try:
        model.load_state_dict(state)
        return model
    except RuntimeError as e:
        print("\nCheckpoint loading failed.")
        print("Reason:")
        print(e)
        raise


# ============================================================
# 11. MNIST CNN Agent with full logging
# ============================================================

class MNISTCNNAgent:

    TRAINING_CONFIG = {
        "base_model":        "SimpleCNN-MNIST",
        "task":              "mnist_digit_classification",
        "dataset":           "torchvision.MNIST",
        "train_split":       "60000_samples",
        "test_split":        "10000_samples",
        "optimizer":         "Adam",
        "learning_rate":     0.001,
        "train_batch_size":  64,
        "eval_batch_size":   1,
        "num_train_epochs":  5,
        "weight_decay":      None,
        "input_shape":       "1x28x28",
        "num_labels":        10,
        "normalization_mean": 0.1307,
        "normalization_std":  0.3081,
        "best_model_metric": "test_accuracy",
        "loss_function":     "CrossEntropyLoss",
    }

    GENERATION_CONFIG = {
        "temperature":      "N/A",
        "top_p":            "N/A",
        "top_k":            "N/A",
        "do_sample":        False,
        "max_new_tokens":   "N/A",
        "decoding_used":    False,
        "inference_mode":   "image_classification",
        "prediction_rule":  "argmax_softmax",
    }

    def __init__(self, classifier_path="simple_cnn_classifier.pth"):
        self.classifier_name = "CNN"
        self.classifier_path = classifier_path
        self.device = DEVICE_EXEC

        self.classifier = SimpleCNNClassifier().to(self.device)
        self.classifier = load_cnn_checkpoint_safely(
            self.classifier,
            classifier_path,
            self.device
        )
        self.classifier.eval()

        self.test_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])

        self.model_param = sum(p.numel() for p in self.classifier.parameters())
        self.trainable_param = sum(
            p.numel() for p in self.classifier.parameters() if p.requires_grad
        )

        self.model_flops = compute_model_flops(self.classifier, self.device)

        self.model_dtype = str(next(self.classifier.parameters()).dtype)
        self.precision_type = "fp16" if "float16" in self.model_dtype else "fp32"
        self.fp16_enabled = "float16" in self.model_dtype
        self.fp32_enabled = "float32" in self.model_dtype

        self.arch_meta = {
            "hidden_sizes":       "Conv2d(1->16) -> MaxPool -> Conv2d(16->32) -> MaxPool -> FC(1568->128->10)",
            "num_hidden_layers":  4,
            "activation":         "ReLU",
            "context_window_size": "N/A",
            "vocab_size":          "N/A",
            "pad_token_id":        "N/A",
        }

        if self.model_flops:
            print(f"Model FLOPs (28x28 input): {self.model_flops:,}")
        else:
            print("FLOPs unavailable.")

    # ------------------------------------------------------------
    # Agent step payloads
    # ------------------------------------------------------------

    def _agent_init_step(self):
        return (
            f"Agent initialized on device {self.device}; "
            f"classifier loaded: {self.classifier_name}; "
            f"params: {self.model_param:,}; "
            f"FLOPs: {self.model_flops}"
        )

    def _agent_task_received_step(self, task):
        return f"Task received from user: {task}"

    def _agent_planning_step(self, task):
        task_lower = task.lower()

        if "test" in task_lower or "classify" in task_lower:
            return {
                "task_supported": True,
                "message": "Task supported. Plan: load MNIST test set and classify each image using CNN."
            }

        return {
            "task_supported": False,
            "message": "Task not supported. This agent only supports MNIST classification."
        }

    # ------------------------------------------------------------
    # Inference
    # ------------------------------------------------------------

    def _infer(self, image_tensor):
        with torch.no_grad():
            logits = self.classifier(image_tensor)
            probs = torch.softmax(logits, dim=1)
            confidence, predicted = torch.max(probs, dim=1)

        return logits, int(predicted.item()), float(confidence.item())

    # ------------------------------------------------------------
    # Log row builder
    # ------------------------------------------------------------

    def _build_log_row(
        self,
        task_id,
        agent_step,
        message,
        step_time,
        true_label=None,
        predicted_label=None,
        logits=None,
        cpu_energy=0,
        gpu_energy=0,
        ram_energy=0,
        total_energy=0,
        emissions_value=0,
        carbon_intensity=None,
        gpu_metrics=None,
        model_accuracy=None,
        model_precision_weighted=None,
        model_recall_weighted=None,
        model_f1_weighted=None,
    ):
        gpu_metrics = gpu_metrics or {}

        correct = None
        confidence_score = None
        logit_margin = None
        entropy = None

        if true_label is not None and predicted_label is not None:
            correct = int(predicted_label) == int(true_label)

        if logits is not None:
            confidence_score, logit_margin, entropy = get_prediction_quality(logits)

        # DNN/CNN image-style token proxy
        input_tokens = 784
        output_tokens = 1
        total_tokens = input_tokens + output_tokens

        tokens_per_second = round(total_tokens / step_time, 4) if step_time > 0 else None

        total_energy = total_energy if total_energy is not None else 0.0
        cpu_energy = cpu_energy if cpu_energy is not None else 0.0
        gpu_energy = gpu_energy if gpu_energy is not None else 0.0
        ram_energy = ram_energy if ram_energy is not None else 0.0

        joules_per_token = 0.0
        energy_per_token_kwh = 0.0
        watts_estimated = 0.0
        gpu_energy_pct = 0.0
        cpu_energy_pct = 0.0

        if total_energy > 0 and total_tokens > 0:
            energy_per_token_kwh = round(total_energy / total_tokens, 12)
            joules_per_token = round((total_energy * 3_600_000) / total_tokens, 6)

            if step_time > 0:
                watts_estimated = round((total_energy * 3_600_000) / step_time, 4)

            gpu_energy_pct = round((gpu_energy / total_energy) * 100, 2)
            cpu_energy_pct = round((cpu_energy / total_energy) * 100, 2)

        cpu_usage = get_cpu_usage()
        ram_usage = get_ram_usage()
        cpu_freq = get_cpu_freq()
        cpu_temp = get_cpu_temp()
        cpu_power_w = get_cpu_power_draw_w()
        cpu_cores_used = get_cpu_cores_used()

        tc = self.TRAINING_CONFIG
        gc = self.GENERATION_CONFIG
        step_cfg = AGENT_STEP_CONFIG.get(agent_step, {})

        return {
            # --- Identity ---
            "timestamp":           time.strftime("%Y-%m-%d %H:%M:%S"),
            "unique_device_id":    DEVICE_UUID,
            "device_short_id":     DEVICE_SHORT,
            "pc_name":             get_hostname(),
            "collection_mode":     "automated",

            # --- Agent step ---
            "task_id":             task_id,
            "sample_index":        task_id - 1 if task_id > 0 else None,
            "agent_step":          agent_step,
            "agent_step_description": step_cfg.get("description"),
            "agent_step_energy_enabled": step_cfg.get("measure_energy"),
            "classifier":          self.classifier_name,
            "message":             message,
            "step_time_seconds":   format(step_time, ".10f"),
            "execution_time_sec":  format(step_time, ".10f"),

            # --- Prediction ---
            "true_label":          true_label,
            "predicted_label":     predicted_label,
            "prediction":          predicted_label,
            "correct":             correct,
            "confidence_score":    confidence_score,
            "logit_margin":        logit_margin,
            "entropy":             entropy,

            # --- Model identity ---
            "model_type":          "SimpleCNN-MNIST",
            "base_model":          tc["base_model"],
            "task":                tc["task"],
            "dataset":             tc["dataset"],

            # --- Parameters ---
            "parameters":          self.model_param,
            "trainable_parameters": self.trainable_param,
            "model_param":         self.model_param,
            "model_flops":         self.model_flops,

            # --- CodeCarbon energy ---
            "cpu_energy_kwh":             cpu_energy,
            "gpu_energy_kwh":             gpu_energy,
            "ram_energy_kwh":             ram_energy,
            "total_energy_kwh":           total_energy,
            "total_emissions_kg":         emissions_value,
            "carbon_intensity_kgco2_kwh": carbon_intensity,

            # --- Efficiency derived ---
            "input_tokens":               input_tokens,
            "output_tokens":              output_tokens,
            "total_tokens":               total_tokens,
            "tokens_per_second":          tokens_per_second,
            "joules_per_token":           joules_per_token,
            "energy_per_token_kwh":       energy_per_token_kwh,
            "watts_estimated":            watts_estimated,
            "gpu_energy_pct_of_total":    gpu_energy_pct,
            "cpu_energy_pct_of_total":    cpu_energy_pct,
            "batch_size_at_inference":    1,

            # --- CPU hardware ---
            "cpu_model":           CPU_MODEL_NAME,
            "cpu_architecture":    CPU_ARCH,
            "cpu_core_count":      CPU_CORE_COUNT,
            "cpu_thread_count":    CPU_THREAD_COUNT,
            "cpu_tdp_w":           CPU_TDP_W,
            "cpu_usage_pct":       cpu_usage,
            "cpu_clock_mhz":       cpu_freq,
            "cpu_temp_c":          cpu_temp,
            "cpu_power_draw_w":    cpu_power_w,
            "cpu_cores_used":      cpu_cores_used,

            # --- GPU hardware ---
            "gpu_model":                get_gpu_name(),
            "gpu_driver_version":       GPU_STATIC["gpu_driver_version"],
            "gpu_compute_capability":   GPU_STATIC["gpu_compute_capability"],
            "gpu_power_limit_w":        GPU_STATIC["gpu_power_limit_w"],
            "gpu_memory_total_mb":      GPU_STATIC["gpu_memory_total_mb"],
            "gpu_power_draw_w":         gpu_metrics.get("gpu_power_draw_w"),
            "gpu_utilization_pct":      gpu_metrics.get("gpu_utilization_pct"),
            "gpu_temp_c":               gpu_metrics.get("gpu_temp_c"),
            "gpu_memory_used_mb":       gpu_metrics.get("gpu_memory_used_mb"),
            "gpu_sm_clock_mhz":         gpu_metrics.get("gpu_sm_clock_mhz"),
            "gpu_memory_clock_mhz":     gpu_metrics.get("gpu_memory_clock_mhz"),
            "cuda_driver_version":      CUDA_DRIVER_VERSION,
            "cuda_available":           torch.cuda.is_available(),
            "device_type":              str(self.device),

            # --- RAM / memory ---
            "ram_usage_pct":        ram_usage,
            "memory_footprint_mb":  get_memory_footprint_mb(),
            "system_ram_total_gb":  SYSTEM_RAM_TOTAL_GB,

            # --- Environment ---
            "os_name":              OS_NAME,
            "os_version":           OS_VERSION,
            "os_architecture":      OS_ARCHITECTURE,
            "os_full_name":         OS_FULL_NAME,
            "python_version":       PYTHON_VERSION,
            "torch_version":        TORCH_VERSION,
            "codecarbon_version":   CODECARBON_VERSION,

            # --- Precision ---
            "model_dtype":          self.model_dtype,
            "precision_type":       self.precision_type,
            "fp16_enabled":         self.fp16_enabled,
            "fp32_enabled":         self.fp32_enabled,

            # --- Architecture ---
            "hidden_sizes":         self.arch_meta["hidden_sizes"],
            "num_hidden_layers":    self.arch_meta["num_hidden_layers"],
            "activation":           self.arch_meta["activation"],
            "context_window_size":  self.arch_meta["context_window_size"],
            "vocab_size":           self.arch_meta["vocab_size"],
            "pad_token_id":         self.arch_meta["pad_token_id"],

            # --- Generation / inference config ---
            "temperature":          gc["temperature"],
            "top_p":                gc["top_p"],
            "top_k":                gc["top_k"],
            "do_sample":            gc["do_sample"],
            "max_new_tokens":       gc["max_new_tokens"],
            "decoding_used":        gc["decoding_used"],
            "inference_mode":       gc["inference_mode"],
            "prediction_rule":      gc["prediction_rule"],

            # --- Training config ---
            "optimizer":            tc["optimizer"],
            "learning_rate":        tc["learning_rate"],
            "train_batch_size":     tc["train_batch_size"],
            "eval_batch_size":      tc["eval_batch_size"],
            "num_train_epochs":     tc["num_train_epochs"],
            "weight_decay":         tc["weight_decay"],
            "input_shape":          tc["input_shape"],
            "num_labels":           tc["num_labels"],
            "normalization_mean":   tc["normalization_mean"],
            "normalization_std":    tc["normalization_std"],
            "best_model_metric":    tc["best_model_metric"],
            "loss_function":        tc["loss_function"],
            "train_split":          tc["train_split"],
            "test_split":           tc["test_split"],

            # --- Final model metrics ---
            "model_accuracy":           model_accuracy,
            "model_precision_weighted": model_precision_weighted,
            "model_recall_weighted":    model_recall_weighted,
            "model_f1_weighted":        model_f1_weighted,
        }

    def _print_log(self, row):
        print(
            f"Task {row['task_id']} | "
            f"{row['agent_step']} | "
            f"Classifier: {row['classifier']} | "
            f"{row['message']} | "
            f"Time: {row['step_time_seconds']} sec | "
            f"Energy: {row['total_energy_kwh']} kWh | "
            f"Watts: {row['watts_estimated']}"
        )

    def _save_logs(self, logs, log_file):
        df = pd.DataFrame(logs)

        if SAVE_CSV:
            df.to_csv(log_file, index=False)
            print(f"CSV saved to: {log_file}")

        if SAVE_EXCEL:
            xlsx_file = log_file.replace(".csv", ".xlsx")
            df.to_excel(xlsx_file, index=False)
            print(f"Excel saved to: {xlsx_file}")

        return df

    def run(self, task, log_file="mnist_agent_cnn_full_log.csv"):
        all_logs = []

        # =============================================
        # Step 1: [AGENT INIT]
        # =============================================
        (
            msg,
            step_time,
            cpu_energy,
            gpu_energy,
            ram_energy,
            total_energy,
            emissions_value,
            carbon_intensity,
            gpu_snap,
        ) = measure_agent_step(
            self._agent_init_step,
            step_name="agent_init",
            measure_energy=AGENT_STEP_CONFIG["[AGENT INIT]"]["measure_energy"]
        )

        row = self._build_log_row(
            task_id=0,
            agent_step="[AGENT INIT]",
            message=msg,
            step_time=step_time,
            cpu_energy=cpu_energy,
            gpu_energy=gpu_energy,
            ram_energy=ram_energy,
            total_energy=total_energy,
            emissions_value=emissions_value,
            carbon_intensity=carbon_intensity,
            gpu_metrics=gpu_snap,
        )

        all_logs.append(row)
        self._print_log(row)

        # =============================================
        # Step 2: [AGENT TASK RECEIVED]
        # =============================================
        (
            msg,
            step_time,
            cpu_energy,
            gpu_energy,
            ram_energy,
            total_energy,
            emissions_value,
            carbon_intensity,
            gpu_snap,
        ) = measure_agent_step(
            self._agent_task_received_step,
            task,
            step_name="agent_task_received",
            measure_energy=AGENT_STEP_CONFIG["[AGENT TASK RECEIVED]"]["measure_energy"]
        )

        row = self._build_log_row(
            task_id=0,
            agent_step="[AGENT TASK RECEIVED]",
            message=msg,
            step_time=step_time,
            cpu_energy=cpu_energy,
            gpu_energy=gpu_energy,
            ram_energy=ram_energy,
            total_energy=total_energy,
            emissions_value=emissions_value,
            carbon_intensity=carbon_intensity,
            gpu_metrics=gpu_snap,
        )

        all_logs.append(row)
        self._print_log(row)

        # =============================================
        # Step 3: [AGENT PLANNING]
        # =============================================
        (
            planning_result,
            step_time,
            cpu_energy,
            gpu_energy,
            ram_energy,
            total_energy,
            emissions_value,
            carbon_intensity,
            gpu_snap,
        ) = measure_agent_step(
            self._agent_planning_step,
            task,
            step_name="agent_planning",
            measure_energy=AGENT_STEP_CONFIG["[AGENT PLANNING]"]["measure_energy"]
        )

        task_supported = planning_result["task_supported"]
        msg = planning_result["message"]

        row = self._build_log_row(
            task_id=0,
            agent_step="[AGENT PLANNING]",
            message=msg,
            step_time=step_time,
            cpu_energy=cpu_energy,
            gpu_energy=gpu_energy,
            ram_energy=ram_energy,
            total_energy=total_energy,
            emissions_value=emissions_value,
            carbon_intensity=carbon_intensity,
            gpu_metrics=gpu_snap,
        )

        all_logs.append(row)
        self._print_log(row)

        if not task_supported:
            self._save_logs(all_logs, log_file)
            return {
                "status": "failed",
                "message": msg,
                "log_file": log_file,
            }

        # =============================================
        # Step 4: [AGENT ACTION]
        # =============================================
        return self._run_action_loop(all_logs, log_file)

    def _run_action_loop(self, all_logs, log_file):
        print("\n[AGENT ACTION]")
        print("Loading MNIST test dataset and classifying images...\n")

        test_dataset = datasets.MNIST(
            root="./data",
            train=False,
            download=True,
            transform=self.test_transform
        )
        NUM_TEST_SAMPLES = 10
        if NUM_TEST_SAMPLES is None:
            total_tasks = len(test_dataset)
        else:
            total_tasks = min(int(NUM_TEST_SAMPLES), len(test_dataset))

        print(f"Configured NUM_TEST_SAMPLES: {NUM_TEST_SAMPLES}")
        print(f"Total samples to classify: {total_tasks}")

        correct_count = 0
        wrong_count = 0
        total_action_time = 0.0

        class_correct = [0] * 10
        class_total = [0] * 10

        true_ids = []
        pred_ids = []

        overall_start = time.perf_counter()

        for task_index in range(total_tasks):
            task_id = task_index + 1

            image, true_label = test_dataset[task_index]
            true_label = int(true_label)

            image_tensor = image.unsqueeze(0).to(self.device)

            (
                (logits, predicted_label, confidence_value),
                exec_time,
                cpu_energy,
                gpu_energy,
                ram_energy,
                total_energy,
                emissions_value,
                carbon_intensity,
                gpu_snap,
            ) = run_with_energy_tracking(
                self._infer,
                image_tensor,
                output_dir="./energy_logs"
            )

            total_action_time += exec_time

            correct = predicted_label == true_label

            if correct:
                correct_count += 1
                class_correct[true_label] += 1
            else:
                wrong_count += 1

            class_total[true_label] += 1

            true_ids.append(true_label)
            pred_ids.append(predicted_label)

            msg = (
                f"Classified MNIST image. "
                f"True: {true_label}, Predicted: {predicted_label}, "
                f"Confidence: {confidence_value:.6f}, Correct: {correct}"
            )

            row = self._build_log_row(
                task_id=task_id,
                agent_step="[AGENT ACTION]",
                message=msg,
                step_time=exec_time,
                true_label=true_label,
                predicted_label=predicted_label,
                logits=logits,
                cpu_energy=cpu_energy,
                gpu_energy=gpu_energy,
                ram_energy=ram_energy,
                total_energy=total_energy,
                emissions_value=emissions_value,
                carbon_intensity=carbon_intensity,
                gpu_metrics=gpu_snap,
            )

            all_logs.append(row)
            self._print_log(row)

            if (task_index + 1) % 500 == 0 or (task_index + 1) == total_tasks:
                print(f"  Completed {task_index + 1}/{total_tasks} samples")

        overall_runtime = time.perf_counter() - overall_start

        if SKLEARN_AVAILABLE:
            accuracy = accuracy_score(true_ids, pred_ids)
            prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
                true_ids,
                pred_ids,
                average="weighted",
                zero_division=0
            )
        else:
            accuracy = correct_count / total_tasks
            prec_w = None
            rec_w = None
            f1_w = None

        for row in all_logs:
            row["model_accuracy"] = accuracy
            row["model_precision_weighted"] = prec_w
            row["model_recall_weighted"] = rec_w
            row["model_f1_weighted"] = f1_w

        self._save_logs(all_logs, log_file)

        per_class_accuracy = {
            d: round(100 * class_correct[d] / class_total[d], 2)
            if class_total[d] > 0 else 0.0
            for d in range(10)
        }

        print("\n" + "=" * 60)
        print("Inference logging complete")
        print("=" * 60)
        print(f"Rows   : {len(all_logs)}")
        print(f"Columns: {len(all_logs[0]) if all_logs else 0}")
        print("\nFinal metrics:")
        print(f"  Accuracy           : {accuracy:.4f}")

        if prec_w is not None:
            print(f"  Weighted Precision : {prec_w:.4f}")
            print(f"  Weighted Recall    : {rec_w:.4f}")
            print(f"  Weighted F1        : {f1_w:.4f}")

        return {
            "status": "success",
            "task": "MNIST classification with full CNN agent log",
            "classifier": self.classifier_name,
            "total_images_classified": total_tasks,
            "correct_predictions": correct_count,
            "wrong_predictions": wrong_count,
            "test_accuracy_percent": round(accuracy * 100, 2),
            "weighted_precision": round(prec_w, 4) if prec_w is not None else None,
            "weighted_recall": round(rec_w, 4) if rec_w is not None else None,
            "weighted_f1": round(f1_w, 4) if f1_w is not None else None,
            "total_action_time_seconds": format(total_action_time, ".10f"),
            "average_action_time_per_image_seconds": format(
                total_action_time / total_tasks,
                ".10f"
            ),
            "overall_runtime_seconds": format(overall_runtime, ".10f"),
            "log_file": log_file.replace(".csv", ".xlsx") if SAVE_EXCEL else log_file,
            "per_class_accuracy_percent": per_class_accuracy,
        }


# ============================================================
# 12. Main
# ============================================================

if __name__ == "__main__":
    classifier_path = "simple_cnn_classifier.pth"

    if not os.path.exists(classifier_path):
        print("No saved CNN classifier found. Training a new CNN classifier.")
        train_cnn_classifier(model_path=classifier_path)
    else:
        print("Saved Simple CNN classifier found. Checking compatibility.")

    try:
        agent = MNISTCNNAgent(classifier_path=classifier_path)
    except RuntimeError:
        print("\nExisting CNN checkpoint is incompatible with this CNN class.")
        print("Deleting old checkpoint and retraining a fresh CNN classifier.\n")

        try:
            os.remove(classifier_path)
        except Exception as e:
            print(f"Could not delete old checkpoint: {e}")

        train_cnn_classifier(model_path=classifier_path)
        agent = MNISTCNNAgent(classifier_path=classifier_path)

    output = agent.run(
        task="Classify all 10000 MNIST test images using Simple CNN",
        log_file="mnist_agent_cnn_full_log.csv",
    )

    print("\n[AGENT FINAL OUTPUT]")
    print(output)

    try:
        if NVML_AVAILABLE:
            pynvml.nvmlShutdown()
    except Exception:
        pass


Saved Simple CNN classifier found. Checking compatibility.
Model FLOPs (28x28 input): 1,218,048
Task 0 | [AGENT INIT] | Classifier: CNN | Agent initialized on device cuda; classifier loaded: CNN; params: 206,922; FLOPs: 1218048 | Time: 0.0000194998 sec | Energy: 1.2130969388928905e-06 kWh | Watts: 223958.6637
Task 0 | [AGENT TASK RECEIVED] | Classifier: CNN | Task received from user: Classify all 10000 MNIST test images using Simple CNN | Time: 0.0000023001 sec | Energy: 1.178495257614524e-06 kWh | Watts: 1844493.8682
Task 0 | [AGENT PLANNING] | Classifier: CNN | Task supported. Plan: load MNIST test set and classify each image using CNN. | Time: 0.0000034000 sec | Energy: 7.617783060090409e-07 kWh | Watts: 806582.6529

[AGENT ACTION]
Loading MNIST test dataset and classifying images...

Configured NUM_TEST_SAMPLES: 10
Total samples to classify: 10
Task 1 | [AGENT ACTION] | Classifier: CNN | Classified MNIST image. True: 7, Predicted: 7, Confidence: 1.000000, Correct: True | Time: 0.00

# LLM

In [ ]:
import os
import sys
import time
import socket
import platform
import hashlib

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import psutil

from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms


# ============================================================
# 0A. RUN CONFIGURATION
# ============================================================

# Device configuration:
# "auto" = CUDA if available, otherwise CPU
# "cpu"  = force CPU
# "cuda" = force CUDA if available
DEVICE_MODE = "auto"

# Number of MNIST test samples to run.
# Use None for all 10,000 samples.
# Use a small number like 10 for debugging.
NUM_TEST_SAMPLES = 10

# Save outputs
SAVE_EXCEL = True
SAVE_CSV = False

# Measure CodeCarbon energy for each agent step.
# Very short steps may still produce extremely small or zero energy values.
AGENT_STEP_CONFIG = {
    "[AGENT INIT]": {
        "measure_energy": True,
        "description": "Initialize the Tiny LLM agent and load model metadata."
    },
    "[AGENT TASK RECEIVED]": {
        "measure_energy": True,
        "description": "Receive the user task."
    },
    "[AGENT PLANNING]": {
        "measure_energy": True,
        "description": "Check task type and build the execution plan."
    },
    "[AGENT ACTION]": {
        "measure_energy": True,
        "description": "Run per-sample MNIST token-sequence classification."
    },
}


def resolve_device():
    if DEVICE_MODE.lower() == "cpu":
        return torch.device("cpu")

    if DEVICE_MODE.lower() == "cuda":
        if torch.cuda.is_available():
            return torch.device("cuda")
        print("CUDA requested but not available. Falling back to CPU.")
        return torch.device("cpu")

    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE_EXEC = resolve_device()


def get_os_full_name():
    system = platform.system()
    architecture = platform.machine()

    if system == "Windows":
        try:
            import winreg
            key = winreg.OpenKey(
                winreg.HKEY_LOCAL_MACHINE,
                r"SOFTWARE\Microsoft\Windows NT\CurrentVersion"
            )
            product_name = winreg.QueryValueEx(key, "ProductName")[0]
            display_version = winreg.QueryValueEx(key, "DisplayVersion")[0]
            current_build = winreg.QueryValueEx(key, "CurrentBuild")[0]
            return f"{product_name} {display_version} Build {current_build} {architecture}"
        except Exception:
            return f"Windows {platform.release()} {architecture}"

    if system == "Linux":
        os_info = {}
        try:
            with open("/etc/os-release", "r", encoding="utf-8") as file:
                for line in file:
                    if "=" in line:
                        key, value = line.strip().split("=", 1)
                        os_info[key] = value.strip('"')
        except Exception:
            pass

        pretty_name = os_info.get("PRETTY_NAME")
        name = os_info.get("NAME")
        version = os_info.get("VERSION")
        version_id = os_info.get("VERSION_ID")
        distro_id = os_info.get("ID")

        if pretty_name:
            return f"{pretty_name} {architecture}"
        if name and version:
            return f"{name} {version} {architecture}"
        if name and version_id:
            return f"{name} {version_id} {architecture}"
        if distro_id:
            return f"{distro_id} {platform.release()} {architecture}"
        return f"Linux {platform.release()} {architecture}"

    if system == "Darwin":
        return f"macOS {platform.mac_ver()[0]} {architecture}"

    return f"{system} {platform.release()} {architecture}"


# ============================================================
# 0. Safe optional imports
# ============================================================

try:
    from codecarbon import EmissionsTracker
    import codecarbon
    CODECARBON_AVAILABLE = True
    CODECARBON_VERSION = codecarbon.__version__
except Exception:
    EmissionsTracker = None
    CODECARBON_AVAILABLE = False
    CODECARBON_VERSION = "unavailable"
    print("CodeCarbon not available. Energy values will be set to 0.")

try:
    import pynvml
    pynvml.nvmlInit()
    NVML_AVAILABLE = True
    NVML_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0) if torch.cuda.is_available() else None
except Exception:
    NVML_AVAILABLE = False
    NVML_HANDLE = None

try:
    import cpuinfo
    _CPU_INFO = cpuinfo.get_cpu_info()
    CPU_MODEL_NAME = _CPU_INFO.get("brand_raw", "Unknown")
    CPU_ARCH = _CPU_INFO.get("arch", platform.machine())
    CPU_TDP_W = None
except Exception:
    CPU_MODEL_NAME = "Unknown"
    CPU_ARCH = platform.machine()
    CPU_TDP_W = None

try:
    from fvcore.nn import FlopCountAnalysis
    FVCORE_AVAILABLE = True
except Exception:
    FlopCountAnalysis = None
    FVCORE_AVAILABLE = False

try:
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    SKLEARN_AVAILABLE = True
except Exception:
    accuracy_score = None
    precision_recall_fscore_support = None
    SKLEARN_AVAILABLE = False
    print("sklearn not available. Final precision/recall/F1 will be set to None.")


TORCH_VERSION = torch.__version__
PYTHON_VERSION = sys.version.split()[0]
OS_NAME = platform.system()
OS_VERSION = platform.version()
OS_ARCHITECTURE = platform.machine()
OS_FULL_NAME = get_os_full_name()
SYSTEM_RAM_TOTAL_GB = round(psutil.virtual_memory().total / (1024 ** 3), 2)
CPU_CORE_COUNT = psutil.cpu_count(logical=False)
CPU_THREAD_COUNT = psutil.cpu_count(logical=True)


# ============================================================
# 1. Stable device ID
# ============================================================

def make_stable_device_id():
    raw = f"{socket.gethostname()}-{platform.system()}-{platform.machine()}-{CPU_MODEL_NAME}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


DEVICE_UUID = make_stable_device_id()
DEVICE_SHORT = DEVICE_UUID[:8]


# ============================================================
# 2. GPU static info
# ============================================================

def _get_cuda_driver_version():
    if not NVML_AVAILABLE:
        return None

    try:
        driver = pynvml.nvmlSystemGetDriverVersion()
        return driver.decode("utf-8") if isinstance(driver, bytes) else driver
    except Exception:
        return None


CUDA_DRIVER_VERSION = _get_cuda_driver_version()


def _get_gpu_static():
    defaults = {
        "gpu_power_limit_w": None,
        "gpu_driver_version": CUDA_DRIVER_VERSION,
        "gpu_memory_total_mb": None,
        "gpu_compute_capability": None,
    }

    if not NVML_AVAILABLE or NVML_HANDLE is None:
        return defaults

    try:
        power_limit_mw = pynvml.nvmlDeviceGetPowerManagementLimit(NVML_HANDLE)
        mem_info = pynvml.nvmlDeviceGetMemoryInfo(NVML_HANDLE)
        cc_major, cc_minor = pynvml.nvmlDeviceGetCudaComputeCapability(NVML_HANDLE)

        return {
            "gpu_power_limit_w": round(power_limit_mw / 1000.0, 1),
            "gpu_driver_version": CUDA_DRIVER_VERSION,
            "gpu_memory_total_mb": round(mem_info.total / (1024 ** 2), 2),
            "gpu_compute_capability": f"{cc_major}.{cc_minor}",
        }
    except Exception:
        return defaults


GPU_STATIC = _get_gpu_static()


# ============================================================
# 3. Hardware helpers
# ============================================================

def get_hostname():
    return socket.gethostname()


def get_gpu_name():
    return torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU"


def get_cpu_usage():
    return psutil.cpu_percent(interval=None)


def get_ram_usage():
    return psutil.virtual_memory().percent


def get_cpu_freq():
    freq = psutil.cpu_freq()
    return round(freq.current, 2) if freq else None


def get_memory_footprint_mb():
    process = psutil.Process(os.getpid())
    return round(process.memory_info().rss / (1024 * 1024), 4)


def get_gpu_metrics():
    null = {
        "gpu_power_draw_w": None,
        "gpu_utilization_pct": None,
        "gpu_temp_c": None,
        "gpu_memory_used_mb": None,
        "gpu_sm_clock_mhz": None,
        "gpu_memory_clock_mhz": None,
    }

    if not NVML_AVAILABLE or NVML_HANDLE is None:
        return null

    try:
        power_mw = pynvml.nvmlDeviceGetPowerUsage(NVML_HANDLE)
        util = pynvml.nvmlDeviceGetUtilizationRates(NVML_HANDLE)
        temp = pynvml.nvmlDeviceGetTemperature(NVML_HANDLE, pynvml.NVML_TEMPERATURE_GPU)
        mem_info = pynvml.nvmlDeviceGetMemoryInfo(NVML_HANDLE)
        sm_clock = pynvml.nvmlDeviceGetClockInfo(NVML_HANDLE, pynvml.NVML_CLOCK_SM)
        mem_clock = pynvml.nvmlDeviceGetClockInfo(NVML_HANDLE, pynvml.NVML_CLOCK_MEM)

        return {
            "gpu_power_draw_w": round(power_mw / 1000.0, 2),
            "gpu_utilization_pct": util.gpu,
            "gpu_temp_c": temp,
            "gpu_memory_used_mb": round(mem_info.used / (1024 ** 2), 2),
            "gpu_sm_clock_mhz": sm_clock,
            "gpu_memory_clock_mhz": mem_clock,
        }
    except Exception:
        return null


def get_cpu_temp():
    try:
        temps = psutil.sensors_temperatures()
        if not temps:
            return None

        for key in ("coretemp", "k10temp", "cpu_thermal", "acpitz"):
            if key in temps:
                values = [e.current for e in temps[key] if e.current and e.current > 0]
                if values:
                    return round(sum(values) / len(values), 1)
    except Exception:
        pass

    return None


def get_cpu_power_draw_w():
    rapl_path = "/sys/class/powercap/intel-rapl/intel-rapl:0/energy_uj"

    try:
        if os.path.exists(rapl_path):
            with open(rapl_path) as f:
                e1 = int(f.read().strip())

            time.sleep(0.1)

            with open(rapl_path) as f:
                e2 = int(f.read().strip())

            return round((e2 - e1) / 1e6 / 0.1, 2)
    except Exception:
        pass

    return None


def get_cpu_cores_used():
    try:
        return sum(1 for p in psutil.cpu_percent(percpu=True) if p > 1.0)
    except Exception:
        return None


# ============================================================
# 4. FLOPs helper
# ============================================================

def compute_model_flops(model, device, vocab_size, seq_len):
    """
    FLOPs for Tiny LLM using dummy token IDs.
    """
    if not FVCORE_AVAILABLE:
        return None

    try:
        dummy = torch.randint(
            low=0,
            high=vocab_size,
            size=(1, seq_len),
            dtype=torch.long,
            device=device
        )

        fc = FlopCountAnalysis(model, dummy)
        fc.unsupported_ops_warnings(False)
        fc.uncalled_modules_warnings(False)

        return int(fc.total())
    except Exception as e:
        print(f"[FLOPs unavailable] {e}")
        return None


# ============================================================
# 5. Prediction quality helpers
# ============================================================

def get_prediction_quality(logits):
    probs = F.softmax(logits, dim=-1).squeeze()
    confidence = float(probs.max().item())

    top2 = torch.topk(logits.squeeze(), k=2).values
    margin = float((top2[0] - top2[1]).item())

    entropy = float(-(probs * torch.log(probs + 1e-12)).sum().item())

    return round(confidence, 6), round(margin, 6), round(entropy, 6)


# ============================================================
# 6. Energy tracking wrapper
# ============================================================

def run_with_energy_tracking(inference_fn, *args, output_dir="./energy_logs", **kwargs):
    os.makedirs(output_dir, exist_ok=True)

    if CODECARBON_AVAILABLE:
        tracker = EmissionsTracker(
            project_name="mnist_tiny_llm_agent_inference",
            output_dir=output_dir,
            output_file="codecarbon_mnist_tiny_llm_inference_log.csv",
            log_level="error",
            save_to_file=True,
        )

        tracker.start()

        t0 = time.perf_counter()
        result = inference_fn(*args, **kwargs)
        exec_time = time.perf_counter() - t0

        emissions_value = tracker.stop()
        gpu_snap = get_gpu_metrics()

        fd = getattr(tracker, "final_emissions_data", None)

        cpu_energy = getattr(fd, "cpu_energy", 0) if fd else 0
        gpu_energy = getattr(fd, "gpu_energy", 0) if fd else 0
        ram_energy = getattr(fd, "ram_energy", 0) if fd else 0
        total_energy = getattr(fd, "energy_consumed", 0) if fd else 0

        carbon_intensity = None
        if emissions_value and total_energy and total_energy > 0:
            carbon_intensity = round(emissions_value / total_energy, 8)

        return (
            result,
            exec_time,
            cpu_energy,
            gpu_energy,
            ram_energy,
            total_energy,
            emissions_value,
            carbon_intensity,
            gpu_snap,
        )

    t0 = time.perf_counter()
    result = inference_fn(*args, **kwargs)
    exec_time = time.perf_counter() - t0
    gpu_snap = get_gpu_metrics()

    return result, exec_time, 0, 0, 0, 0, 0, None, gpu_snap



def measure_agent_step(
    step_fn,
    *args,
    output_dir="./energy_logs",
    step_name="agent_step",
    measure_energy=True,
    **kwargs
):
    """
    Measure non-inference agent steps with CodeCarbon.

    Used for:
    [AGENT INIT]
    [AGENT TASK RECEIVED]
    [AGENT PLANNING]
    """
    os.makedirs(output_dir, exist_ok=True)

    if CODECARBON_AVAILABLE and measure_energy:
        tracker = EmissionsTracker(
            project_name=f"mnist_tiny_llm_agent_{step_name}",
            output_dir=output_dir,
            output_file=f"codecarbon_mnist_tiny_llm_{step_name}.csv",
            log_level="error",
            save_to_file=True,
        )

        tracker.start()

        t0 = time.perf_counter()
        result = step_fn(*args, **kwargs)
        exec_time = time.perf_counter() - t0

        emissions_value = tracker.stop()
        gpu_snap = get_gpu_metrics()

        fd = getattr(tracker, "final_emissions_data", None)

        cpu_energy = getattr(fd, "cpu_energy", 0) if fd else 0
        gpu_energy = getattr(fd, "gpu_energy", 0) if fd else 0
        ram_energy = getattr(fd, "ram_energy", 0) if fd else 0
        total_energy = getattr(fd, "energy_consumed", 0) if fd else 0

        carbon_intensity = None
        if emissions_value and total_energy and total_energy > 0:
            carbon_intensity = round(emissions_value / total_energy, 8)

        return (
            result,
            exec_time,
            cpu_energy,
            gpu_energy,
            ram_energy,
            total_energy,
            emissions_value,
            carbon_intensity,
            gpu_snap,
        )

    t0 = time.perf_counter()
    result = step_fn(*args, **kwargs)
    exec_time = time.perf_counter() - t0
    gpu_snap = get_gpu_metrics()

    return result, exec_time, 0, 0, 0, 0, 0, None, gpu_snap


# ============================================================
# 7. MNIST Token Dataset
# ============================================================

class MNISTTokenDataset(Dataset):
    def __init__(self, train=True, root="./data"):
        self.mnist = datasets.MNIST(
            root=root,
            train=train,
            download=True,
            transform=transforms.ToTensor()
        )

        self.token_to_id = {
            "<pad>": 0,
            "<bos>": 1,
            "classify_digit": 2,
            "<ans>": 3,
        }

        for i in range(16):
            self.token_to_id[f"P{i}"] = len(self.token_to_id)

        self.id_to_token = {v: k for k, v in self.token_to_id.items()}

        self.seq_len = 52
        self.vocab_size = len(self.token_to_id)

    def image_to_patch_tokens(self, image_tensor):
        image = image_tensor.squeeze(0)
        tokens = []

        for i in range(0, 28, 4):
            for j in range(0, 28, 4):
                patch = image[i:i + 4, j:j + 4]
                avg_val = patch.mean().item()
                bin_id = min(int(avg_val * 16), 15)
                tokens.append(f"P{bin_id}")

        return tokens

    def build_input_ids(self, image_tensor):
        patch_tokens = self.image_to_patch_tokens(image_tensor)
        sequence = ["<bos>", "classify_digit"] + patch_tokens + ["<ans>"]
        input_ids = [self.token_to_id[token] for token in sequence]
        return torch.tensor(input_ids, dtype=torch.long)

    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, idx):
        image, label = self.mnist[idx]
        input_ids = self.build_input_ids(image)
        label_tensor = torch.tensor(label, dtype=torch.long)
        return input_ids, label_tensor


# ============================================================
# 8. Tiny LLM Classifier
# ============================================================

class TinyLLMClassifier(nn.Module):
    def __init__(
        self,
        vocab_size,
        seq_len,
        embed_dim=64,
        num_heads=4,
        num_layers=2,
        ff_dim=128,
        num_classes=10,
        dropout=0.1
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(seq_len, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.norm = nn.LayerNorm(embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, input_ids):
        batch_size, seq_len = input_ids.shape

        positions = torch.arange(seq_len, device=input_ids.device)
        positions = positions.unsqueeze(0).expand(batch_size, seq_len)

        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        x = self.transformer(x)

        x = self.norm(x[:, -1, :])
        logits = self.classifier(x)

        return logits


# ============================================================
# 9. Train Tiny LLM Classifier
# ============================================================

def evaluate_tiny_llm(model, test_loader, device):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for input_ids, labels in test_loader:
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            outputs = model(input_ids)
            _, predicted = torch.max(outputs, dim=1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total


def train_tiny_llm_classifier(model_path="tiny_llm_mnist_classifier.pth"):
    train_dataset = MNISTTokenDataset(train=True, root="./data")
    test_dataset = MNISTTokenDataset(train=False, root="./data")

    train_loader = DataLoader(
        train_dataset,
        batch_size=128,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=512,
        shuffle=False
    )

    device = DEVICE_EXEC
    print(f"Training device: {device}")

    model = TinyLLMClassifier(
        vocab_size=train_dataset.vocab_size,
        seq_len=train_dataset.seq_len,
        embed_dim=64,
        num_heads=4,
        num_layers=2,
        ff_dim=128,
        num_classes=10,
        dropout=0.1
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

    epochs = 5

    print("\nTraining Tiny LLM MNIST Classifier...")

    for epoch in range(epochs):
        model.train()

        total_loss = 0.0
        correct = 0
        total = 0

        for input_ids, labels in train_loader:
            input_ids = input_ids.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(input_ids)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            _, predicted = torch.max(outputs, dim=1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        avg_loss = total_loss / len(train_loader)
        train_accuracy = 100 * correct / total

        test_accuracy = evaluate_tiny_llm(
            model=model,
            test_loader=test_loader,
            device=device
        )

        print(
            f"Epoch [{epoch + 1}/{epochs}] | "
            f"Loss: {avg_loss:.4f} | "
            f"Train Accuracy: {train_accuracy:.2f}% | "
            f"Test Accuracy: {test_accuracy:.2f}%"
        )

    checkpoint = {
        "model_state_dict": model.state_dict(),
        "vocab_size": train_dataset.vocab_size,
        "seq_len": train_dataset.seq_len,
        "token_to_id": train_dataset.token_to_id,
        "id_to_token": train_dataset.id_to_token,
        "embed_dim": 64,
        "num_heads": 4,
        "num_layers": 2,
        "ff_dim": 128,
        "num_classes": 10,
        "dropout": 0.1,
    }

    torch.save(checkpoint, model_path)

    print(f"\nTiny LLM classifier saved as: {model_path}")


# ============================================================
# 10. MNIST Tiny LLM Agent with full logging
# ============================================================

class MNISTTinyLLMAgent:

    TRAINING_CONFIG = {
        "base_model":        "TinyLLM-MNIST",
        "task":              "mnist_digit_classification",
        "dataset":           "torchvision.MNIST_tokenized_4x4_patches",
        "train_split":       "60000_samples",
        "test_split":        "10000_samples",
        "optimizer":         "AdamW",
        "learning_rate":     0.001,
        "train_batch_size":  128,
        "eval_batch_size":   1,
        "num_train_epochs":  5,
        "weight_decay":      0.01,
        "input_shape":       "token_sequence_length_52",
        "num_labels":        10,
        "normalization_mean": None,
        "normalization_std":  None,
        "best_model_metric": "test_accuracy",
        "loss_function":     "CrossEntropyLoss",
    }

    GENERATION_CONFIG = {
        "temperature":      None,
        "top_p":            None,
        "top_k":            None,
        "do_sample":        False,
        "max_new_tokens":   None,
        "decoding_used":    False,
        "inference_mode":   "token_sequence_classification",
        "prediction_rule":  "argmax_softmax",
    }

    def __init__(self, model_path="tiny_llm_mnist_classifier.pth"):
        self.classifier_name = "Tiny LLM"
        self.model_path = model_path

        self.device = DEVICE_EXEC

        checkpoint = torch.load(model_path, map_location=self.device)

        self.vocab_size = checkpoint["vocab_size"]
        self.seq_len = checkpoint["seq_len"]
        self.embed_dim = checkpoint.get("embed_dim", 64)
        self.num_heads = checkpoint.get("num_heads", 4)
        self.num_layers = checkpoint.get("num_layers", 2)
        self.ff_dim = checkpoint.get("ff_dim", 128)
        self.num_classes = checkpoint.get("num_classes", 10)
        self.dropout = checkpoint.get("dropout", 0.1)

        self.classifier = TinyLLMClassifier(
            vocab_size=self.vocab_size,
            seq_len=self.seq_len,
            embed_dim=self.embed_dim,
            num_heads=self.num_heads,
            num_layers=self.num_layers,
            ff_dim=self.ff_dim,
            num_classes=self.num_classes,
            dropout=self.dropout
        ).to(self.device)

        self.classifier.load_state_dict(checkpoint["model_state_dict"])
        self.classifier.eval()

        self.test_dataset = MNISTTokenDataset(train=False, root="./data")

        self.model_param = sum(p.numel() for p in self.classifier.parameters())
        self.trainable_param = sum(
            p.numel() for p in self.classifier.parameters() if p.requires_grad
        )

        self.model_flops = compute_model_flops(
            self.classifier,
            self.device,
            vocab_size=self.vocab_size,
            seq_len=self.seq_len
        )

        self.model_dtype = str(next(self.classifier.parameters()).dtype)
        self.precision_type = "fp16" if "float16" in self.model_dtype else "fp32"
        self.fp16_enabled = "float16" in self.model_dtype
        self.fp32_enabled = "float32" in self.model_dtype

        self.arch_meta = {
            "hidden_sizes": (
                f"vocab={self.vocab_size}, seq_len={self.seq_len}, "
                f"embed_dim={self.embed_dim}, layers={self.num_layers}, "
                f"heads={self.num_heads}, ff_dim={self.ff_dim}, classes=10"
            ),
            "num_hidden_layers": self.num_layers,
            "activation": "GELU",
            "context_window_size": self.seq_len,
            "vocab_size": self.vocab_size,
            "pad_token_id": 0,
        }

        if self.model_flops:
            print(f"Model FLOPs (token input): {self.model_flops:,}")
        else:
            print("FLOPs unavailable.")

    def _agent_init_step(self):
        return (
            f"Agent initialized on device {self.device}; "
            f"classifier loaded: {self.classifier_name}; "
            f"params: {self.model_param:,}; "
            f"FLOPs: {self.model_flops}"
        )

    def _agent_task_received_step(self, task):
        return f"Task received from user: {task}"

    def _agent_planning_step(self, task):
        task_lower = task.lower()

        if "test" in task_lower or "classify" in task_lower or "mnist" in task_lower:
            return {
                "task_supported": True,
                "message": (
                    "Task supported. Plan: load MNIST token test set and classify "
                    "each tokenized image using Tiny LLM."
                )
            }

        return {
            "task_supported": False,
            "message": "Task not supported. This agent only supports MNIST classification."
        }

    def _infer(self, input_ids):
        with torch.no_grad():
            logits = self.classifier(input_ids)
            probs = torch.softmax(logits, dim=1)
            confidence, predicted = torch.max(probs, dim=1)

        return logits, int(predicted.item()), float(confidence.item())

    def _build_log_row(
        self,
        task_id,
        agent_step,
        message,
        step_time,
        true_label=None,
        predicted_label=None,
        logits=None,
        cpu_energy=0,
        gpu_energy=0,
        ram_energy=0,
        total_energy=0,
        emissions_value=0,
        carbon_intensity=None,
        gpu_metrics=None,
        model_accuracy=None,
        model_precision_weighted=None,
        model_recall_weighted=None,
        model_f1_weighted=None,
    ):
        gpu_metrics = gpu_metrics or {}

        correct = None
        confidence_score = None
        logit_margin = None
        entropy = None

        if true_label is not None and predicted_label is not None:
            correct = int(predicted_label) == int(true_label)

        if logits is not None:
            confidence_score, logit_margin, entropy = get_prediction_quality(logits)

        input_tokens = self.seq_len
        output_tokens = 1
        total_tokens = input_tokens + output_tokens

        tokens_per_second = round(total_tokens / step_time, 4) if step_time > 0 else None

        joules_per_token = 0.0
        energy_per_token_kwh = 0.0
        watts_estimated = 0.0
        gpu_energy_pct = 0.0
        cpu_energy_pct = 0.0

        if total_energy and total_energy > 0:
            energy_per_token_kwh = round(total_energy / total_tokens, 12)
            joules_per_token = round((total_energy * 3_600_000) / total_tokens, 6)

            if step_time > 0:
                watts_estimated = round((total_energy * 3_600_000) / step_time, 4)

            gpu_energy_pct = round((gpu_energy / total_energy) * 100, 2)
            cpu_energy_pct = round((cpu_energy / total_energy) * 100, 2)

        cpu_usage = get_cpu_usage()
        ram_usage = get_ram_usage()
        cpu_freq = get_cpu_freq()
        cpu_temp = get_cpu_temp()
        cpu_power_w = get_cpu_power_draw_w()
        cpu_cores_used = get_cpu_cores_used()

        tc = self.TRAINING_CONFIG
        gc = self.GENERATION_CONFIG
        step_cfg = AGENT_STEP_CONFIG.get(agent_step, {})

        return {
            "timestamp":           time.strftime("%Y-%m-%d %H:%M:%S"),
            "unique_device_id":    DEVICE_UUID,
            "device_short_id":     DEVICE_SHORT,
            "pc_name":             get_hostname(),
            "collection_mode":     "automated",

            "task_id":             task_id,
            "agent_step":          agent_step,
            "agent_step_description": step_cfg.get("description"),
            "agent_step_energy_enabled": step_cfg.get("measure_energy"),
            "classifier":          self.classifier_name,
            "message":             message,
            "step_time_seconds":   format(step_time, ".10f"),
            "execution_time_sec":  format(step_time, ".10f"),

            "true_label":          true_label,
            "predicted_label":     predicted_label,
            "correct":             correct,
            "confidence_score":    confidence_score,
            "logit_margin":        logit_margin,
            "entropy":             entropy,

            "model_type":          "TinyLLM-MNIST",
            "base_model":          tc["base_model"],
            "task":                tc["task"],
            "dataset":             tc["dataset"],

            "parameters":          self.model_param,
            "trainable_parameters": self.trainable_param,
            "model_param":         self.model_param,
            "model_flops":         self.model_flops,

            "cpu_energy_kwh":             cpu_energy,
            "gpu_energy_kwh":             gpu_energy,
            "ram_energy_kwh":             ram_energy,
            "total_energy_kwh":           total_energy,
            "total_emissions_kg":         emissions_value,
            "carbon_intensity_kgco2_kwh": carbon_intensity,

            "input_tokens":               input_tokens,
            "output_tokens":              output_tokens,
            "total_tokens":               total_tokens,
            "tokens_per_second":          tokens_per_second,
            "joules_per_token":           joules_per_token,
            "energy_per_token_kwh":       energy_per_token_kwh,
            "watts_estimated":            watts_estimated,
            "gpu_energy_pct_of_total":    gpu_energy_pct,
            "cpu_energy_pct_of_total":    cpu_energy_pct,
            "batch_size_at_inference":    1,

            "cpu_model":           CPU_MODEL_NAME,
            "cpu_architecture":    CPU_ARCH,
            "cpu_core_count":      CPU_CORE_COUNT,
            "cpu_thread_count":    CPU_THREAD_COUNT,
            "cpu_tdp_w":           CPU_TDP_W,
            "cpu_usage_pct":       cpu_usage,
            "cpu_clock_mhz":       cpu_freq,
            "cpu_temp_c":          cpu_temp,
            "cpu_power_draw_w":    cpu_power_w,
            "cpu_cores_used":      cpu_cores_used,

            "gpu_model":                get_gpu_name(),
            "gpu_driver_version":       GPU_STATIC["gpu_driver_version"],
            "gpu_compute_capability":   GPU_STATIC["gpu_compute_capability"],
            "gpu_power_limit_w":        GPU_STATIC["gpu_power_limit_w"],
            "gpu_memory_total_mb":      GPU_STATIC["gpu_memory_total_mb"],
            "gpu_power_draw_w":         gpu_metrics.get("gpu_power_draw_w"),
            "gpu_utilization_pct":      gpu_metrics.get("gpu_utilization_pct"),
            "gpu_temp_c":               gpu_metrics.get("gpu_temp_c"),
            "gpu_memory_used_mb":       gpu_metrics.get("gpu_memory_used_mb"),
            "gpu_sm_clock_mhz":         gpu_metrics.get("gpu_sm_clock_mhz"),
            "gpu_memory_clock_mhz":     gpu_metrics.get("gpu_memory_clock_mhz"),
            "cuda_driver_version":      CUDA_DRIVER_VERSION,
            "cuda_available":           torch.cuda.is_available(),
            "device_type":              str(self.device),

            "ram_usage_pct":        ram_usage,
            "memory_footprint_mb":  get_memory_footprint_mb(),
            "system_ram_total_gb":  SYSTEM_RAM_TOTAL_GB,

            "os_name":              OS_NAME,
            "os_version":           OS_VERSION,

            "os_architecture":      platform.machine(),
            "os_full_name":         OS_FULL_NAME,

            "python_version":       PYTHON_VERSION,
            "torch_version":        TORCH_VERSION,
            "codecarbon_version":   CODECARBON_VERSION,

            "model_dtype":          self.model_dtype,
            "precision_type":       self.precision_type,
            "fp16_enabled":         self.fp16_enabled,
            "fp32_enabled":         self.fp32_enabled,

            "hidden_sizes":         self.arch_meta["hidden_sizes"],
            "num_hidden_layers":    self.arch_meta["num_hidden_layers"],
            "activation":           self.arch_meta["activation"],
            "context_window_size":  self.arch_meta["context_window_size"],
            "vocab_size":           self.arch_meta["vocab_size"],
            "pad_token_id":         self.arch_meta["pad_token_id"],

            "temperature":          gc["temperature"],
            "top_p":                gc["top_p"],
            "top_k":                gc["top_k"],
            "do_sample":            gc["do_sample"],
            "max_new_tokens":       gc["max_new_tokens"],
            "decoding_used":        gc["decoding_used"],
            "inference_mode":       gc["inference_mode"],
            "prediction_rule":      gc["prediction_rule"],

            "optimizer":            tc["optimizer"],
            "learning_rate":        tc["learning_rate"],
            "train_batch_size":     tc["train_batch_size"],
            "eval_batch_size":      tc["eval_batch_size"],
            "num_train_epochs":     tc["num_train_epochs"],
            "weight_decay":         tc["weight_decay"],
            "input_shape":          tc["input_shape"],
            "num_labels":           tc["num_labels"],
            "normalization_mean":   tc["normalization_mean"],
            "normalization_std":    tc["normalization_std"],
            "best_model_metric":    tc["best_model_metric"],
            "loss_function":        tc["loss_function"],
            "train_split":          tc["train_split"],
            "test_split":           tc["test_split"],

            "model_accuracy":           model_accuracy,
            "model_precision_weighted": model_precision_weighted,
            "model_recall_weighted":    model_recall_weighted,
            "model_f1_weighted":        model_f1_weighted,
        }

    def _print_log(self, row):
        print(
            f"Task {row['task_id']} | "
            f"{row['agent_step']} | "
            f"Classifier: {row['classifier']} | "
            f"{row['message']} | "
            f"Time: {row['step_time_seconds']} sec | "
            f"Energy: {row['total_energy_kwh']} kWh | "
            f"Watts: {row['watts_estimated']}"
        )

    def _save_logs(self, logs, log_file):
        df = pd.DataFrame(logs)

        if SAVE_CSV:
            df.to_csv(log_file, index=False)
            print(f"CSV saved to: {log_file}")

        if SAVE_EXCEL:
            xlsx_file = log_file.replace(".csv", ".xlsx")
            df.to_excel(xlsx_file, index=False)
            print(f"Excel saved to: {xlsx_file}")

        return df

    def run(self, task, log_file="mnist_agent_tiny_llm_full_log.csv"):
        all_logs = []

        # =============================================
        # Step 1: [AGENT INIT] with CodeCarbon
        # =============================================
        (
            msg,
            step_time,
            cpu_energy,
            gpu_energy,
            ram_energy,
            total_energy,
            emissions_value,
            carbon_intensity,
            gpu_snap,
        ) = measure_agent_step(
            self._agent_init_step,
            step_name="agent_init",
            measure_energy=AGENT_STEP_CONFIG["[AGENT INIT]"]["measure_energy"]
        )

        row = self._build_log_row(
            task_id=0,
            agent_step="[AGENT INIT]",
            message=msg,
            step_time=step_time,
            cpu_energy=cpu_energy,
            gpu_energy=gpu_energy,
            ram_energy=ram_energy,
            total_energy=total_energy,
            emissions_value=emissions_value,
            carbon_intensity=carbon_intensity,
            gpu_metrics=gpu_snap,
        )

        all_logs.append(row)
        self._print_log(row)

        # =============================================
        # Step 2: [AGENT TASK RECEIVED] with CodeCarbon
        # =============================================
        (
            msg,
            step_time,
            cpu_energy,
            gpu_energy,
            ram_energy,
            total_energy,
            emissions_value,
            carbon_intensity,
            gpu_snap,
        ) = measure_agent_step(
            self._agent_task_received_step,
            task,
            step_name="agent_task_received",
            measure_energy=AGENT_STEP_CONFIG["[AGENT TASK RECEIVED]"]["measure_energy"]
        )

        row = self._build_log_row(
            task_id=0,
            agent_step="[AGENT TASK RECEIVED]",
            message=msg,
            step_time=step_time,
            cpu_energy=cpu_energy,
            gpu_energy=gpu_energy,
            ram_energy=ram_energy,
            total_energy=total_energy,
            emissions_value=emissions_value,
            carbon_intensity=carbon_intensity,
            gpu_metrics=gpu_snap,
        )

        all_logs.append(row)
        self._print_log(row)

        # =============================================
        # Step 3: [AGENT PLANNING] with CodeCarbon
        # =============================================
        (
            planning_result,
            step_time,
            cpu_energy,
            gpu_energy,
            ram_energy,
            total_energy,
            emissions_value,
            carbon_intensity,
            gpu_snap,
        ) = measure_agent_step(
            self._agent_planning_step,
            task,
            step_name="agent_planning",
            measure_energy=AGENT_STEP_CONFIG["[AGENT PLANNING]"]["measure_energy"]
        )

        task_supported = planning_result["task_supported"]
        msg = planning_result["message"]

        row = self._build_log_row(
            task_id=0,
            agent_step="[AGENT PLANNING]",
            message=msg,
            step_time=step_time,
            cpu_energy=cpu_energy,
            gpu_energy=gpu_energy,
            ram_energy=ram_energy,
            total_energy=total_energy,
            emissions_value=emissions_value,
            carbon_intensity=carbon_intensity,
            gpu_metrics=gpu_snap,
        )

        all_logs.append(row)
        self._print_log(row)

        if not task_supported:
            self._save_logs(all_logs, log_file)
            return {
                "status": "failed",
                "message": msg,
                "log_file": log_file
            }

        return self._run_action_loop(all_logs, log_file)

    def _run_action_loop(self, all_logs, log_file):
        print("\n[AGENT ACTION]")
        print("Loading MNIST token test dataset and classifying all 10,000 samples...\n")

        total_tasks = len(self.test_dataset) if NUM_TEST_SAMPLES is None else min(int(NUM_TEST_SAMPLES), len(self.test_dataset))

        correct_count = 0
        wrong_count = 0
        total_action_time = 0.0

        class_correct = [0] * 10
        class_total = [0] * 10

        true_ids = []
        pred_ids = []

        overall_start = time.perf_counter()

        for task_index in range(total_tasks):
            task_id = task_index + 1

            input_ids, true_label = self.test_dataset[task_index]
            true_label = int(true_label)

            input_tensor = input_ids.unsqueeze(0).to(self.device)

            (
                (logits, predicted_label, confidence_value),
                exec_time,
                cpu_energy,
                gpu_energy,
                ram_energy,
                total_energy,
                emissions_value,
                carbon_intensity,
                gpu_snap,
            ) = run_with_energy_tracking(self._infer, input_tensor)

            total_action_time += exec_time

            correct = predicted_label == true_label

            if correct:
                correct_count += 1
                class_correct[true_label] += 1
            else:
                wrong_count += 1

            class_total[true_label] += 1

            true_ids.append(true_label)
            pred_ids.append(predicted_label)

            msg = (
                f"Classified tokenized MNIST image. "
                f"True: {true_label}, Predicted: {predicted_label}, "
                f"Confidence: {confidence_value:.6f}, Correct: {correct}"
            )

            row = self._build_log_row(
                task_id=task_id,
                agent_step="[AGENT ACTION]",
                message=msg,
                step_time=exec_time,
                true_label=true_label,
                predicted_label=predicted_label,
                logits=logits,
                cpu_energy=cpu_energy,
                gpu_energy=gpu_energy,
                ram_energy=ram_energy,
                total_energy=total_energy,
                emissions_value=emissions_value,
                carbon_intensity=carbon_intensity,
                gpu_metrics=gpu_snap,
            )

            all_logs.append(row)
            self._print_log(row)

            if (task_index + 1) % 500 == 0 or (task_index + 1) == total_tasks:
                print(f"  Completed {task_index + 1}/{total_tasks} samples")

        overall_runtime = time.perf_counter() - overall_start

        if SKLEARN_AVAILABLE:
            accuracy = accuracy_score(true_ids, pred_ids)
            prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
                true_ids,
                pred_ids,
                average="weighted",
                zero_division=0
            )
        else:
            accuracy = correct_count / total_tasks
            prec_w = None
            rec_w = None
            f1_w = None

        for row in all_logs:
            row["model_accuracy"] = accuracy
            row["model_precision_weighted"] = prec_w
            row["model_recall_weighted"] = rec_w
            row["model_f1_weighted"] = f1_w

        self._save_logs(all_logs, log_file)

        per_class_accuracy = {
            d: round(100 * class_correct[d] / class_total[d], 2)
            if class_total[d] > 0 else 0.0
            for d in range(10)
        }

        print("\n" + "=" * 60)
        print("Tiny LLM inference logging complete")
        print("=" * 60)
        print(f"Rows   : {len(all_logs)}")
        print(f"Columns: {len(all_logs[0]) if all_logs else 0}")
        print("\nFinal metrics:")
        print(f"  Accuracy           : {accuracy:.4f}")

        if prec_w is not None:
            print(f"  Weighted Precision : {prec_w:.4f}")
            print(f"  Weighted Recall    : {rec_w:.4f}")
            print(f"  Weighted F1        : {f1_w:.4f}")

        return {
            "status": "success",
            "task": "MNIST classification with full Tiny LLM agent log",
            "classifier": self.classifier_name,
            "total_images_classified": total_tasks,
            "correct_predictions": correct_count,
            "wrong_predictions": wrong_count,
            "test_accuracy_percent": round(accuracy * 100, 2),
            "weighted_precision": round(prec_w, 4) if prec_w is not None else None,
            "weighted_recall": round(rec_w, 4) if rec_w is not None else None,
            "weighted_f1": round(f1_w, 4) if f1_w is not None else None,
            "total_action_time_seconds": format(total_action_time, ".10f"),
            "average_action_time_per_image_seconds": format(
                total_action_time / total_tasks,
                ".10f"
            ),
            "overall_runtime_seconds": format(overall_runtime, ".10f"),
            "log_file": log_file.replace(".csv", ".xlsx") if SAVE_EXCEL else log_file,
            "per_class_accuracy_percent": per_class_accuracy,
        }



# ============================================================
# 11. Checkpoint helper
# ============================================================

def checkpoint_is_compatible(model_path):
    """
    Returns True only if the saved checkpoint has the expected TinyLLM fields.
    This prevents crashing when an older or different checkpoint is present.
    """
    if not os.path.exists(model_path):
        return False

    try:
        checkpoint = torch.load(model_path, map_location="cpu")
        required_keys = {
            "model_state_dict",
            "vocab_size",
            "seq_len",
            "embed_dim",
            "num_heads",
            "num_layers",
            "ff_dim",
            "num_classes",
        }
        return isinstance(checkpoint, dict) and required_keys.issubset(set(checkpoint.keys()))
    except Exception:
        return False


# ============================================================
# 11. Main
# ============================================================

if __name__ == "__main__":
    model_path = "tiny_llm_mnist_classifier.pth"

    if not checkpoint_is_compatible(model_path):
        if os.path.exists(model_path):
            print("Existing Tiny LLM checkpoint is incompatible. Retraining a fresh model.")
            try:
                os.remove(model_path)
            except Exception:
                pass
        train_tiny_llm_classifier(model_path=model_path)
    else:
        print("Saved compatible Tiny LLM classifier found. Skipping training.")

    agent = MNISTTinyLLMAgent(model_path=model_path)

    output = agent.run(
        task="Classify all 10000 MNIST test images using Tiny LLM",
        log_file="mnist_agent_tiny_llm_full_log.csv",
    )

    print("\n[AGENT FINAL OUTPUT]")
    print(output)

    try:
        if NVML_AVAILABLE:
            pynvml.nvmlShutdown()
    except Exception:
        pass

Saved compatible Tiny LLM classifier found. Skipping training.
Model FLOPs (token input): 3,475,392
Task 0 | [AGENT INIT] | Classifier: Tiny LLM | Agent initialized on device cuda; classifier loaded: Tiny LLM; params: 72,330; FLOPs: 3475392 | Time: 0.0000173000 sec | Energy: 8.741188600348929e-07 kWh | Watts: 181897.4052
Task 0 | [AGENT TASK RECEIVED] | Classifier: Tiny LLM | Task received from user: Classify all 10000 MNIST test images using Tiny LLM | Time: 0.0000020000 sec | Energy: 6.572278113000922e-07 kWh | Watts: 1183001.0524
Task 0 | [AGENT PLANNING] | Classifier: Tiny LLM | Task supported. Plan: load MNIST token test set and classify each tokenized image using Tiny LLM. | Time: 0.0000048999 sec | Energy: 1.3050683191977443e-06 kWh | Watts: 958841.1832

[AGENT ACTION]
Loading MNIST token test dataset and classifying all 10,000 samples...

Task 1 | [AGENT ACTION] | Classifier: Tiny LLM | Classified tokenized MNIST image. True: 7, Predicted: 7, Confidence: 0.998773, Correct: True

# VLM

In [ ]:
import os
import sys
import time
import socket
import platform
import hashlib

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import psutil

from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms



# ============================================================
# 0A. RUN CONFIGURATION
# ============================================================

# Device configuration:
# "auto" = CUDA if available, otherwise CPU
# "cpu"  = force CPU
# "cuda" = force CUDA if available
DEVICE_MODE = "auto"

# TinyVLM debugging:
# Use 10 for quick test.
# Use None for all 10,000 MNIST test samples.
NUM_TEST_SAMPLES = 10

# Save outputs
SAVE_EXCEL = True
SAVE_CSV = False

# Measure energy for all agent steps using CodeCarbon.
# Very short steps can produce tiny or zero values, but they will be tracked.
AGENT_STEP_CONFIG = {
    "[AGENT INIT]": {
        "measure_energy": True,
        "description": "Initialize the TinyVLM agent and load model metadata."
    },
    "[AGENT TASK RECEIVED]": {
        "measure_energy": True,
        "description": "Receive the user task."
    },
    "[AGENT PLANNING]": {
        "measure_energy": True,
        "description": "Check task type and build the execution plan."
    },
    "[AGENT ACTION]": {
        "measure_energy": True,
        "description": "Run per-sample MNIST classification using TinyVLM."
    },
}


def resolve_device():
    if DEVICE_MODE.lower() == "cpu":
        return torch.device("cpu")

    if DEVICE_MODE.lower() == "cuda":
        if torch.cuda.is_available():
            return torch.device("cuda")
        print("CUDA requested but not available. Falling back to CPU.")
        return torch.device("cpu")

    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE_EXEC = resolve_device()


def get_os_full_name():
    system = platform.system()
    architecture = platform.machine()

    if system == "Windows":
        try:
            import winreg
            key = winreg.OpenKey(
                winreg.HKEY_LOCAL_MACHINE,
                r"SOFTWARE\Microsoft\Windows NT\CurrentVersion"
            )
            product_name = winreg.QueryValueEx(key, "ProductName")[0]
            display_version = winreg.QueryValueEx(key, "DisplayVersion")[0]
            current_build = winreg.QueryValueEx(key, "CurrentBuild")[0]
            return f"{product_name} {display_version} Build {current_build} {architecture}"
        except Exception:
            return f"Windows {platform.release()} {architecture}"

    if system == "Linux":
        os_info = {}
        try:
            with open("/etc/os-release", "r") as file:
                for line in file:
                    if "=" in line:
                        key, value = line.strip().split("=", 1)
                        os_info[key] = value.strip('"')
        except Exception:
            pass

        pretty_name = os_info.get("PRETTY_NAME")
        name = os_info.get("NAME")
        version = os_info.get("VERSION")
        version_id = os_info.get("VERSION_ID")
        distro_id = os_info.get("ID")

        if pretty_name:
            return f"{pretty_name} {architecture}"
        if name and version:
            return f"{name} {version} {architecture}"
        if name and version_id:
            return f"{name} {version_id} {architecture}"
        if distro_id:
            return f"{distro_id} {platform.release()} {architecture}"
        return f"Linux {platform.release()} {architecture}"

    if system == "Darwin":
        return f"macOS {platform.mac_ver()[0]} {architecture}"

    return f"{system} {platform.release()} {architecture}"

# ============================================================
# 0. Safe optional imports
# ============================================================

try:
    from codecarbon import EmissionsTracker
    import codecarbon
    CODECARBON_AVAILABLE = True
    CODECARBON_VERSION = codecarbon.__version__
except Exception:
    EmissionsTracker = None
    CODECARBON_AVAILABLE = False
    CODECARBON_VERSION = "unavailable"
    print("CodeCarbon not available. Energy values will be set to 0.")

try:
    import pynvml
    pynvml.nvmlInit()
    NVML_AVAILABLE = True
    NVML_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0) if torch.cuda.is_available() else None
except Exception:
    NVML_AVAILABLE = False
    NVML_HANDLE = None

try:
    import cpuinfo
    _CPU_INFO = cpuinfo.get_cpu_info()
    CPU_MODEL_NAME = _CPU_INFO.get("brand_raw", "Unknown")
    CPU_ARCH = _CPU_INFO.get("arch", platform.machine())
    CPU_TDP_W = None
except Exception:
    CPU_MODEL_NAME = "Unknown"
    CPU_ARCH = platform.machine()
    CPU_TDP_W = None

try:
    from fvcore.nn import FlopCountAnalysis
    FVCORE_AVAILABLE = True
except Exception:
    FlopCountAnalysis = None
    FVCORE_AVAILABLE = False

try:
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    SKLEARN_AVAILABLE = True
except Exception:
    accuracy_score = None
    precision_recall_fscore_support = None
    SKLEARN_AVAILABLE = False
    print("sklearn not available. Final precision/recall/F1 will be set to None.")


TORCH_VERSION = torch.__version__
PYTHON_VERSION = sys.version.split()[0]
OS_NAME = platform.system()
OS_VERSION = platform.version()
OS_ARCHITECTURE = platform.machine()
OS_FULL_NAME = get_os_full_name()
SYSTEM_RAM_TOTAL_GB = round(psutil.virtual_memory().total / (1024 ** 3), 2)
CPU_CORE_COUNT = psutil.cpu_count(logical=False)
CPU_THREAD_COUNT = psutil.cpu_count(logical=True)


# ============================================================
# 1. Stable device ID
# ============================================================

def make_stable_device_id():
    raw = f"{socket.gethostname()}-{platform.system()}-{platform.machine()}-{CPU_MODEL_NAME}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


DEVICE_UUID = make_stable_device_id()
DEVICE_SHORT = DEVICE_UUID[:8]


# ============================================================
# 2. GPU static info
# ============================================================

def _get_cuda_driver_version():
    if not NVML_AVAILABLE:
        return None

    try:
        driver = pynvml.nvmlSystemGetDriverVersion()
        return driver.decode("utf-8") if isinstance(driver, bytes) else driver
    except Exception:
        return None


CUDA_DRIVER_VERSION = _get_cuda_driver_version()


def _get_gpu_static():
    defaults = {
        "gpu_power_limit_w": None,
        "gpu_driver_version": CUDA_DRIVER_VERSION,
        "gpu_memory_total_mb": None,
        "gpu_compute_capability": None,
    }

    if not NVML_AVAILABLE or NVML_HANDLE is None:
        return defaults

    try:
        power_limit_mw = pynvml.nvmlDeviceGetPowerManagementLimit(NVML_HANDLE)
        mem_info = pynvml.nvmlDeviceGetMemoryInfo(NVML_HANDLE)
        cc_major, cc_minor = pynvml.nvmlDeviceGetCudaComputeCapability(NVML_HANDLE)

        return {
            "gpu_power_limit_w": round(power_limit_mw / 1000.0, 1),
            "gpu_driver_version": CUDA_DRIVER_VERSION,
            "gpu_memory_total_mb": round(mem_info.total / (1024 ** 2), 2),
            "gpu_compute_capability": f"{cc_major}.{cc_minor}",
        }
    except Exception:
        return defaults


GPU_STATIC = _get_gpu_static()


# ============================================================
# 3. Hardware helpers
# ============================================================

def get_hostname():
    return socket.gethostname()


def get_gpu_name():
    return torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU"


def get_cpu_usage():
    return psutil.cpu_percent(interval=None)


def get_ram_usage():
    return psutil.virtual_memory().percent


def get_cpu_freq():
    freq = psutil.cpu_freq()
    return round(freq.current, 2) if freq else None


def get_memory_footprint_mb():
    process = psutil.Process(os.getpid())
    return round(process.memory_info().rss / (1024 * 1024), 4)


def get_gpu_metrics():
    null = {
        "gpu_power_draw_w": None,
        "gpu_utilization_pct": None,
        "gpu_temp_c": None,
        "gpu_memory_used_mb": None,
        "gpu_sm_clock_mhz": None,
        "gpu_memory_clock_mhz": None,
    }

    if not NVML_AVAILABLE or NVML_HANDLE is None:
        return null

    try:
        power_mw = pynvml.nvmlDeviceGetPowerUsage(NVML_HANDLE)
        util = pynvml.nvmlDeviceGetUtilizationRates(NVML_HANDLE)
        temp = pynvml.nvmlDeviceGetTemperature(NVML_HANDLE, pynvml.NVML_TEMPERATURE_GPU)
        mem_info = pynvml.nvmlDeviceGetMemoryInfo(NVML_HANDLE)
        sm_clock = pynvml.nvmlDeviceGetClockInfo(NVML_HANDLE, pynvml.NVML_CLOCK_SM)
        mem_clock = pynvml.nvmlDeviceGetClockInfo(NVML_HANDLE, pynvml.NVML_CLOCK_MEM)

        return {
            "gpu_power_draw_w": round(power_mw / 1000.0, 2),
            "gpu_utilization_pct": util.gpu,
            "gpu_temp_c": temp,
            "gpu_memory_used_mb": round(mem_info.used / (1024 ** 2), 2),
            "gpu_sm_clock_mhz": sm_clock,
            "gpu_memory_clock_mhz": mem_clock,
        }
    except Exception:
        return null


def get_cpu_temp():
    try:
        temps = psutil.sensors_temperatures()
        if not temps:
            return None

        for key in ("coretemp", "k10temp", "cpu_thermal", "acpitz"):
            if key in temps:
                values = [e.current for e in temps[key] if e.current and e.current > 0]
                if values:
                    return round(sum(values) / len(values), 1)
    except Exception:
        pass

    return None


def get_cpu_power_draw_w():
    rapl_path = "/sys/class/powercap/intel-rapl/intel-rapl:0/energy_uj"

    try:
        if os.path.exists(rapl_path):
            with open(rapl_path) as f:
                e1 = int(f.read().strip())

            time.sleep(0.1)

            with open(rapl_path) as f:
                e2 = int(f.read().strip())

            return round((e2 - e1) / 1e6 / 0.1, 2)
    except Exception:
        pass

    return None


def get_cpu_cores_used():
    try:
        return sum(1 for p in psutil.cpu_percent(percpu=True) if p > 1.0)
    except Exception:
        return None


# ============================================================
# 4. FLOPs helper
# ============================================================

def compute_model_flops(model, device, vocab_size):
    """
    FLOPs for TinyVLM using one image and 10 text prompts.
    """
    if not FVCORE_AVAILABLE:
        return None

    try:
        dummy_image = torch.ones((1, 1, 28, 28), dtype=torch.float32, device=device)
        dummy_text = torch.randint(
            low=0,
            high=vocab_size,
            size=(10, 2),
            dtype=torch.long,
            device=device
        )

        fc = FlopCountAnalysis(model, (dummy_image, dummy_text))
        fc.unsupported_ops_warnings(False)
        fc.uncalled_modules_warnings(False)

        return int(fc.total())
    except Exception as e:
        print(f"[FLOPs unavailable] {e}")
        return None


# ============================================================
# 5. Prediction quality helpers
# ============================================================

def get_prediction_quality(logits):
    probs = F.softmax(logits, dim=-1).squeeze()
    confidence = float(probs.max().item())

    top2 = torch.topk(logits.squeeze(), k=2).values
    margin = float((top2[0] - top2[1]).item())

    entropy = float(-(probs * torch.log(probs + 1e-12)).sum().item())

    return round(confidence, 6), round(margin, 6), round(entropy, 6)


# ============================================================
# 6. Energy tracking wrappers
# ============================================================

def _extract_energy_data(tracker, emissions_value):
    fd = getattr(tracker, "final_emissions_data", None)

    cpu_energy = getattr(fd, "cpu_energy", 0) if fd else 0
    gpu_energy = getattr(fd, "gpu_energy", 0) if fd else 0
    ram_energy = getattr(fd, "ram_energy", 0) if fd else 0
    total_energy = getattr(fd, "energy_consumed", 0) if fd else 0

    carbon_intensity = None
    if emissions_value and total_energy and total_energy > 0:
        carbon_intensity = round(emissions_value / total_energy, 8)

    return cpu_energy, gpu_energy, ram_energy, total_energy, carbon_intensity


def measure_agent_step(
    step_fn,
    *args,
    output_dir="./energy_logs",
    step_name="agent_step",
    measure_energy=True,
    **kwargs
):
    """
    Measures any agent step using CodeCarbon, not only inference.

    Used for:
    [AGENT INIT]
    [AGENT TASK RECEIVED]
    [AGENT PLANNING]
    [AGENT ACTION]
    """

    os.makedirs(output_dir, exist_ok=True)

    if CODECARBON_AVAILABLE and measure_energy:
        tracker = EmissionsTracker(
            project_name=f"mnist_tiny_vlm_{step_name}",
            output_dir=output_dir,
            output_file=f"codecarbon_mnist_tiny_vlm_{step_name}.csv",
            log_level="error",
            save_to_file=True,
        )

        tracker.start()

        t0 = time.perf_counter()
        result = step_fn(*args, **kwargs)
        exec_time = time.perf_counter() - t0

        emissions_value = tracker.stop()
        gpu_snap = get_gpu_metrics()

        cpu_energy, gpu_energy, ram_energy, total_energy, carbon_intensity = _extract_energy_data(
            tracker,
            emissions_value
        )

        return (
            result,
            exec_time,
            cpu_energy,
            gpu_energy,
            ram_energy,
            total_energy,
            emissions_value,
            carbon_intensity,
            gpu_snap,
        )

    t0 = time.perf_counter()
    result = step_fn(*args, **kwargs)
    exec_time = time.perf_counter() - t0
    gpu_snap = get_gpu_metrics()

    return result, exec_time, 0, 0, 0, 0, 0, None, gpu_snap


def run_with_energy_tracking(inference_fn, *args, output_dir="./energy_logs", **kwargs):
    """
    Backward-compatible wrapper for inference-only calls.
    It now uses measure_agent_step internally.
    """
    return measure_agent_step(
        inference_fn,
        *args,
        output_dir=output_dir,
        step_name="agent_action_inference",
        measure_energy=AGENT_STEP_CONFIG["[AGENT ACTION]"]["measure_energy"],
        **kwargs
    )


# ============================================================
# 7. Tiny VLM Dataset
# ============================================================

class MNISTVLMTrainDataset(Dataset):
    def __init__(self, train=True, root="./data"):
        self.mnist = datasets.MNIST(
            root=root,
            train=train,
            download=True,
            transform=transforms.ToTensor()
        )

    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, idx):
        image, label = self.mnist[idx]
        return image, label


# ============================================================
# 8. Tiny VLM Components
# ============================================================

class TinyImageEncoder(nn.Module):
    def __init__(self, embed_dim=64):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )

        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, embed_dim)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.proj(x)
        return F.normalize(x, dim=-1)


class TinyTextEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, text_hidden_dim=64):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, text_hidden_dim)
        self.proj = nn.Linear(text_hidden_dim, embed_dim)

    def forward(self, token_ids):
        x = self.token_embedding(token_ids)
        x = x.mean(dim=1)
        x = self.proj(x)
        return F.normalize(x, dim=-1)


class TinyVLM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, text_hidden_dim=64):
        super().__init__()

        self.image_encoder = TinyImageEncoder(embed_dim=embed_dim)

        self.text_encoder = TinyTextEncoder(
            vocab_size=vocab_size,
            embed_dim=embed_dim,
            text_hidden_dim=text_hidden_dim
        )

        self.logit_scale = nn.Parameter(torch.tensor(1.0))

    def encode_image(self, images):
        return self.image_encoder(images)

    def encode_text(self, token_ids):
        return self.text_encoder(token_ids)

    def forward(self, images, token_ids):
        image_features = self.encode_image(images)
        text_features = self.encode_text(token_ids)

        scale = self.logit_scale.exp()
        logits = scale * image_features @ text_features.T

        return logits


class VLMTextProcessor:
    def __init__(self):
        self.token_to_id = {
            "<pad>": 0,
            "digit": 1,
            "0": 2,
            "1": 3,
            "2": 4,
            "3": 5,
            "4": 6,
            "5": 7,
            "6": 8,
            "7": 9,
            "8": 10,
            "9": 11,
        }

        self.id_to_token = {v: k for k, v in self.token_to_id.items()}
        self.vocab_size = len(self.token_to_id)
        self.seq_len = 2

    def label_to_prompt_tokens(self, label):
        return ["digit", str(label)]

    def prompt_tokens_to_ids(self, tokens):
        ids = [self.token_to_id[t] for t in tokens]
        return torch.tensor(ids, dtype=torch.long)

    def build_label_token_ids(self, label):
        tokens = self.label_to_prompt_tokens(label)
        return self.prompt_tokens_to_ids(tokens)

    def build_all_class_token_ids(self):
        all_ids = []

        for label in range(10):
            all_ids.append(self.build_label_token_ids(label))

        return torch.stack(all_ids, dim=0)


# ============================================================
# 9. Train Tiny VLM
# ============================================================

def evaluate_tiny_vlm(model, test_loader, text_processor, device):
    model.eval()

    correct = 0
    total = 0

    class_token_ids = text_processor.build_all_class_token_ids().to(device)

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images, class_token_ids)
            _, predicted = torch.max(logits, dim=1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total


def train_tiny_vlm(model_path="tiny_vlm_mnist_classifier.pth"):
    text_processor = VLMTextProcessor()

    train_dataset = MNISTVLMTrainDataset(train=True, root="./data")
    test_dataset = MNISTVLMTrainDataset(train=False, root="./data")

    train_loader = DataLoader(
        train_dataset,
        batch_size=128,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=512,
        shuffle=False
    )

    device = DEVICE_EXEC
    print(f"Training device: {device}")

    model = TinyVLM(
        vocab_size=text_processor.vocab_size,
        embed_dim=64,
        text_hidden_dim=64
    ).to(device)

    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

    epochs = 5

    print("\nTraining Tiny VLM MNIST Classifier...")

    for epoch in range(epochs):
        model.train()

        total_loss = 0.0
        correct = 0
        total = 0

        class_token_ids = text_processor.build_all_class_token_ids().to(device)

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            logits = model(images, class_token_ids)
            loss = F.cross_entropy(logits, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            _, predicted = torch.max(logits, dim=1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        avg_loss = total_loss / len(train_loader)
        train_accuracy = 100 * correct / total

        test_accuracy = evaluate_tiny_vlm(
            model=model,
            test_loader=test_loader,
            text_processor=text_processor,
            device=device
        )

        print(
            f"Epoch [{epoch + 1}/{epochs}] | "
            f"Loss: {avg_loss:.4f} | "
            f"Train Accuracy: {train_accuracy:.2f}% | "
            f"Test Accuracy: {test_accuracy:.2f}%"
        )

    checkpoint = {
        "model_state_dict": model.state_dict(),
        "vocab_size": text_processor.vocab_size,
        "token_to_id": text_processor.token_to_id,
        "id_to_token": text_processor.id_to_token,
        "seq_len": text_processor.seq_len,
        "embed_dim": 64,
        "text_hidden_dim": 64,
        "num_classes": 10,
    }

    torch.save(checkpoint, model_path)

    print(f"\nTiny VLM classifier saved as: {model_path}")


# ============================================================
# 10. MNIST Tiny VLM Agent with full logging
# ============================================================

class MNISTTinyVLMAgent:

    TRAINING_CONFIG = {
        "base_model":        "TinyVLM-MNIST",
        "task":              "mnist_digit_classification",
        "dataset":           "torchvision.MNIST_image_text_prompts",
        "train_split":       "60000_samples",
        "test_split":        "10000_samples",
        "optimizer":         "AdamW",
        "learning_rate":     0.001,
        "train_batch_size":  128,
        "eval_batch_size":   1,
        "num_train_epochs":  5,
        "weight_decay":      0.01,
        "input_shape":       "image_1x28x28_plus_10_text_prompts",
        "num_labels":        10,
        "normalization_mean": None,
        "normalization_std":  None,
        "best_model_metric": "test_accuracy",
        "loss_function":     "CrossEntropyLoss_on_image_text_similarity",
    }

    GENERATION_CONFIG = {
        "temperature":      "N/A",
        "top_p":            "N/A",
        "top_k":            "N/A",
        "do_sample":        False,
        "max_new_tokens":   "N/A",
        "decoding_used":    False,
        "inference_mode":   "image_text_similarity_classification",
        "prediction_rule":  "argmax_softmax",
    }

    def __init__(self, model_path="tiny_vlm_mnist_classifier.pth"):
        self.classifier_name = "Tiny VLM"
        self.model_path = model_path

        self.device = DEVICE_EXEC

        checkpoint = torch.load(model_path, map_location=self.device)

        self.vocab_size = checkpoint["vocab_size"]
        self.seq_len = checkpoint.get("seq_len", 2)
        self.embed_dim = checkpoint.get("embed_dim", 64)
        self.text_hidden_dim = checkpoint.get("text_hidden_dim", 64)
        self.num_classes = checkpoint.get("num_classes", 10)

        self.text_processor = VLMTextProcessor()
        if "token_to_id" in checkpoint:
            self.text_processor.token_to_id = checkpoint["token_to_id"]
            self.text_processor.id_to_token = {v: k for k, v in self.text_processor.token_to_id.items()}
            self.text_processor.vocab_size = len(self.text_processor.token_to_id)
        if "seq_len" in checkpoint:
            self.text_processor.seq_len = checkpoint["seq_len"]

        self.classifier = TinyVLM(
            vocab_size=self.vocab_size,
            embed_dim=self.embed_dim,
            text_hidden_dim=self.text_hidden_dim
        ).to(self.device)

        try:
            self.classifier.load_state_dict(checkpoint["model_state_dict"])
        except RuntimeError as e:
            raise RuntimeError(
                "The saved tiny_vlm_mnist_classifier.pth checkpoint does not match "
                "the current TinyVLM architecture. Delete the checkpoint and rerun, "
                "or let the main block retrain it."
            ) from e

        self.classifier.eval()

        self.class_token_ids = self.text_processor.build_all_class_token_ids().to(self.device)

        self.test_dataset = MNISTVLMTrainDataset(train=False, root="./data")

        self.model_param = sum(p.numel() for p in self.classifier.parameters())
        self.trainable_param = sum(
            p.numel() for p in self.classifier.parameters() if p.requires_grad
        )

        self.model_flops = compute_model_flops(
            self.classifier,
            self.device,
            vocab_size=self.vocab_size
        )

        self.model_dtype = str(next(self.classifier.parameters()).dtype)
        self.precision_type = "fp16" if "float16" in self.model_dtype else "fp32"
        self.fp16_enabled = "float16" in self.model_dtype
        self.fp32_enabled = "float32" in self.model_dtype

        self.arch_meta = {
            "hidden_sizes": (
                f"image_encoder=Conv(1->16)->Conv(16->32)->MLP(1568->128->{self.embed_dim}); "
                f"text_encoder=Embedding(vocab={self.vocab_size}, hidden={self.text_hidden_dim})"
                f"->Linear({self.text_hidden_dim}->{self.embed_dim}); "
                f"text_prompts=10, prompt_seq_len={self.seq_len}"
            ),
            "num_hidden_layers": 4,
            "activation": "ReLU",
            "context_window_size": self.seq_len,
            "vocab_size": self.vocab_size,
            "pad_token_id": 0,
        }

        if self.model_flops:
            print(f"Model FLOPs (image + text prompts): {self.model_flops:,}")
        else:
            print("FLOPs unavailable.")

    # ------------------------------------------------------------
    # Agent step payloads
    # ------------------------------------------------------------

    def _agent_init_step(self):
        return (
            f"Agent initialized on device {self.device}; "
            f"classifier loaded: {self.classifier_name}; "
            f"params: {self.model_param:,}; "
            f"FLOPs: {self.model_flops}"
        )

    def _agent_task_received_step(self, task):
        return f"Task received from user: {task}"

    def _agent_planning_step(self, task):
        task_lower = task.lower()

        if "test" in task_lower or "classify" in task_lower:
            return {
                "task_supported": True,
                "message": (
                    "Task supported. Plan: load MNIST image test set and classify "
                    "each image using Tiny VLM image-text similarity."
                )
            }

        return {
            "task_supported": False,
            "message": "Task not supported. This agent only supports MNIST classification."
        }

    def _infer(self, image_tensor):
        with torch.no_grad():
            logits = self.classifier(image_tensor, self.class_token_ids)
            probs = torch.softmax(logits, dim=1)
            confidence, predicted = torch.max(probs, dim=1)

        return logits, int(predicted.item()), float(confidence.item())

    def _build_log_row(
        self,
        task_id,
        agent_step,
        message,
        step_time,
        true_label=None,
        predicted_label=None,
        logits=None,
        cpu_energy=0,
        gpu_energy=0,
        ram_energy=0,
        total_energy=0,
        emissions_value=0,
        carbon_intensity=None,
        gpu_metrics=None,
        model_accuracy=None,
        model_precision_weighted=None,
        model_recall_weighted=None,
        model_f1_weighted=None,
    ):
        gpu_metrics = gpu_metrics or {}

        correct = None
        confidence_score = None
        logit_margin = None
        entropy = None

        if true_label is not None and predicted_label is not None:
            correct = int(predicted_label) == int(true_label)

        if logits is not None:
            confidence_score, logit_margin, entropy = get_prediction_quality(logits)

        input_tokens = 784 + (10 * self.seq_len)
        output_tokens = 1
        total_tokens = input_tokens + output_tokens

        tokens_per_second = round(total_tokens / step_time, 4) if step_time > 0 else None

        joules_per_token = 0.0
        energy_per_token_kwh = 0.0
        watts_estimated = 0.0
        gpu_energy_pct = 0.0
        cpu_energy_pct = 0.0

        if total_energy and total_energy > 0:
            energy_per_token_kwh = round(total_energy / total_tokens, 12)
            joules_per_token = round((total_energy * 3_600_000) / total_tokens, 6)

            if step_time > 0:
                watts_estimated = round((total_energy * 3_600_000) / step_time, 4)

            gpu_energy_pct = round((gpu_energy / total_energy) * 100, 2)
            cpu_energy_pct = round((cpu_energy / total_energy) * 100, 2)

        cpu_usage = get_cpu_usage()
        ram_usage = get_ram_usage()
        cpu_freq = get_cpu_freq()
        cpu_temp = get_cpu_temp()
        cpu_power_w = get_cpu_power_draw_w()
        cpu_cores_used = get_cpu_cores_used()

        tc = self.TRAINING_CONFIG
        gc = self.GENERATION_CONFIG
        step_cfg = AGENT_STEP_CONFIG.get(agent_step, {})

        return {
            "timestamp":           time.strftime("%Y-%m-%d %H:%M:%S"),
            "unique_device_id":    DEVICE_UUID,
            "device_short_id":     DEVICE_SHORT,
            "pc_name":             get_hostname(),
            "collection_mode":     "automated",

            "task_id":             task_id,
            "agent_step":          agent_step,
            "agent_step_description": step_cfg.get("description"),
            "agent_step_energy_enabled": step_cfg.get("measure_energy"),
            "classifier":          self.classifier_name,
            "message":             message,
            "step_time_seconds":   format(step_time, ".10f"),

            "true_label":          true_label,
            "predicted_label":     predicted_label,
            "correct":             correct,
            "confidence_score":    confidence_score,
            "logit_margin":        logit_margin,
            "entropy":             entropy,

            "model_type":          "TinyVLM-MNIST",
            "base_model":          tc["base_model"],
            "task":                tc["task"],
            "dataset":             tc["dataset"],

            "parameters":          self.model_param,
            "trainable_parameters": self.trainable_param,
            "model_flops":         self.model_flops,

            "cpu_energy_kwh":             cpu_energy,
            "gpu_energy_kwh":             gpu_energy,
            "ram_energy_kwh":             ram_energy,
            "total_energy_kwh":           total_energy,
            "total_emissions_kg":         emissions_value,
            "carbon_intensity_kgco2_kwh": carbon_intensity,

            "input_tokens":               input_tokens,
            "output_tokens":              output_tokens,
            "total_tokens":               total_tokens,
            "tokens_per_second":          tokens_per_second,
            "joules_per_token":           joules_per_token,
            "energy_per_token_kwh":       energy_per_token_kwh,
            "watts_estimated":            watts_estimated,
            "gpu_energy_pct_of_total":    gpu_energy_pct,
            "cpu_energy_pct_of_total":    cpu_energy_pct,
            "batch_size_at_inference":    1,

            "cpu_model":           CPU_MODEL_NAME,
            "cpu_architecture":    CPU_ARCH,
            "cpu_core_count":      CPU_CORE_COUNT,
            "cpu_thread_count":    CPU_THREAD_COUNT,
            "cpu_tdp_w":           CPU_TDP_W,
            "cpu_usage_pct":       cpu_usage,
            "cpu_clock_mhz":       cpu_freq,
            "cpu_temp_c":          cpu_temp,
            "cpu_power_draw_w":    cpu_power_w,
            "cpu_cores_used":      cpu_cores_used,

            "gpu_model":                get_gpu_name(),
            "gpu_driver_version":       GPU_STATIC["gpu_driver_version"],
            "gpu_compute_capability":   GPU_STATIC["gpu_compute_capability"],
            "gpu_power_limit_w":        GPU_STATIC["gpu_power_limit_w"],
            "gpu_memory_total_mb":      GPU_STATIC["gpu_memory_total_mb"],
            "gpu_power_draw_w":         gpu_metrics.get("gpu_power_draw_w"),
            "gpu_utilization_pct":      gpu_metrics.get("gpu_utilization_pct"),
            "gpu_temp_c":               gpu_metrics.get("gpu_temp_c"),
            "gpu_memory_used_mb":       gpu_metrics.get("gpu_memory_used_mb"),
            "gpu_sm_clock_mhz":         gpu_metrics.get("gpu_sm_clock_mhz"),
            "gpu_memory_clock_mhz":     gpu_metrics.get("gpu_memory_clock_mhz"),
            "cuda_driver_version":      CUDA_DRIVER_VERSION,
            "cuda_available":           torch.cuda.is_available(),
            "device_type":              str(self.device),

            "ram_usage_pct":        ram_usage,
            "memory_footprint_mb":  get_memory_footprint_mb(),
            "system_ram_total_gb":  SYSTEM_RAM_TOTAL_GB,

            "os_name":              OS_NAME,
            "os_version":           OS_VERSION,
            "os_architecture":      OS_ARCHITECTURE,
            "os_full_name":         OS_FULL_NAME,
            "python_version":       PYTHON_VERSION,
            "torch_version":        TORCH_VERSION,
            "codecarbon_version":   CODECARBON_VERSION,

            "model_dtype":          self.model_dtype,
            "precision_type":       self.precision_type,
            "fp16_enabled":         self.fp16_enabled,
            "fp32_enabled":         self.fp32_enabled,

            "hidden_sizes":         self.arch_meta["hidden_sizes"],
            "num_hidden_layers":    self.arch_meta["num_hidden_layers"],
            "activation":           self.arch_meta["activation"],
            "context_window_size":  self.arch_meta["context_window_size"],
            "vocab_size":           self.arch_meta["vocab_size"],
            "pad_token_id":         self.arch_meta["pad_token_id"],

            "temperature":          gc["temperature"],
            "top_p":                gc["top_p"],
            "top_k":                gc["top_k"],
            "do_sample":            gc["do_sample"],
            "max_new_tokens":       gc["max_new_tokens"],
            "decoding_used":        gc["decoding_used"],
            "inference_mode":       gc["inference_mode"],
            "prediction_rule":      gc["prediction_rule"],

            "optimizer":            tc["optimizer"],
            "learning_rate":        tc["learning_rate"],
            "train_batch_size":     tc["train_batch_size"],
            "eval_batch_size":      tc["eval_batch_size"],
            "num_train_epochs":     tc["num_train_epochs"],
            "weight_decay":         tc["weight_decay"],
            "input_shape":          tc["input_shape"],
            "num_labels":           tc["num_labels"],
            "normalization_mean":   tc["normalization_mean"],
            "normalization_std":    tc["normalization_std"],
            "best_model_metric":    tc["best_model_metric"],
            "loss_function":        tc["loss_function"],
            "train_split":          tc["train_split"],
            "test_split":           tc["test_split"],

            "model_accuracy":           model_accuracy,
            "model_precision_weighted": model_precision_weighted,
            "model_recall_weighted":    model_recall_weighted,
            "model_f1_weighted":        model_f1_weighted,
        }

    def _print_log(self, row):
        print(
            f"Task {row['task_id']} | "
            f"{row['agent_step']} | "
            f"Classifier: {row['classifier']} | "
            f"{row['message']} | "
            f"Time: {row['step_time_seconds']} sec | "
            f"Energy: {row['total_energy_kwh']} kWh | "
            f"Watts: {row['watts_estimated']}"
        )

    def _save_logs(self, logs, log_file):
        df = pd.DataFrame(logs)

        if SAVE_CSV:
            df.to_csv(log_file, index=False)
            print(f"CSV saved to: {log_file}")

        if SAVE_EXCEL:
            xlsx_file = log_file.replace(".csv", ".xlsx")
            df.to_excel(xlsx_file, index=False)
            print(f"Excel saved to: {xlsx_file}")

        return df

    def run(self, task, log_file="mnist_agent_tiny_vlm_full_log.csv"):
        all_logs = []

        # =============================================
        # Step 1: [AGENT INIT]
        # =============================================
        (
            msg,
            step_time,
            cpu_energy,
            gpu_energy,
            ram_energy,
            total_energy,
            emissions_value,
            carbon_intensity,
            gpu_snap,
        ) = measure_agent_step(
            self._agent_init_step,
            step_name="agent_init",
            measure_energy=AGENT_STEP_CONFIG["[AGENT INIT]"]["measure_energy"]
        )

        row = self._build_log_row(
            task_id=0,
            agent_step="[AGENT INIT]",
            message=msg,
            step_time=step_time,
            cpu_energy=cpu_energy,
            gpu_energy=gpu_energy,
            ram_energy=ram_energy,
            total_energy=total_energy,
            emissions_value=emissions_value,
            carbon_intensity=carbon_intensity,
            gpu_metrics=gpu_snap,
        )

        all_logs.append(row)
        self._print_log(row)

        # =============================================
        # Step 2: [AGENT TASK RECEIVED]
        # =============================================
        (
            msg,
            step_time,
            cpu_energy,
            gpu_energy,
            ram_energy,
            total_energy,
            emissions_value,
            carbon_intensity,
            gpu_snap,
        ) = measure_agent_step(
            self._agent_task_received_step,
            task,
            step_name="agent_task_received",
            measure_energy=AGENT_STEP_CONFIG["[AGENT TASK RECEIVED]"]["measure_energy"]
        )

        row = self._build_log_row(
            task_id=0,
            agent_step="[AGENT TASK RECEIVED]",
            message=msg,
            step_time=step_time,
            cpu_energy=cpu_energy,
            gpu_energy=gpu_energy,
            ram_energy=ram_energy,
            total_energy=total_energy,
            emissions_value=emissions_value,
            carbon_intensity=carbon_intensity,
            gpu_metrics=gpu_snap,
        )

        all_logs.append(row)
        self._print_log(row)

        # =============================================
        # Step 3: [AGENT PLANNING]
        # =============================================
        (
            planning_result,
            step_time,
            cpu_energy,
            gpu_energy,
            ram_energy,
            total_energy,
            emissions_value,
            carbon_intensity,
            gpu_snap,
        ) = measure_agent_step(
            self._agent_planning_step,
            task,
            step_name="agent_planning",
            measure_energy=AGENT_STEP_CONFIG["[AGENT PLANNING]"]["measure_energy"]
        )

        task_supported = planning_result["task_supported"]
        msg = planning_result["message"]

        row = self._build_log_row(
            task_id=0,
            agent_step="[AGENT PLANNING]",
            message=msg,
            step_time=step_time,
            cpu_energy=cpu_energy,
            gpu_energy=gpu_energy,
            ram_energy=ram_energy,
            total_energy=total_energy,
            emissions_value=emissions_value,
            carbon_intensity=carbon_intensity,
            gpu_metrics=gpu_snap,
        )

        all_logs.append(row)
        self._print_log(row)

        if not task_supported:
            self._save_logs(all_logs, log_file)
            return {
                "status": "failed",
                "message": msg,
                "log_file": log_file
            }

        return self._run_action_loop(all_logs, log_file)

    def _run_action_loop(self, all_logs, log_file):
        print("\n[AGENT ACTION]")
        print("Loading MNIST image test dataset and classifying all 10,000 samples...\n")

        if NUM_TEST_SAMPLES is None:
            total_tasks = len(self.test_dataset)
        else:
            total_tasks = min(int(NUM_TEST_SAMPLES), len(self.test_dataset))

        correct_count = 0
        wrong_count = 0
        total_action_time = 0.0

        class_correct = [0] * 10
        class_total = [0] * 10

        true_ids = []
        pred_ids = []

        overall_start = time.perf_counter()

        for task_index in range(total_tasks):
            task_id = task_index + 1

            image, true_label = self.test_dataset[task_index]
            true_label = int(true_label)

            image_tensor = image.unsqueeze(0).to(self.device)

            (
                (logits, predicted_label, confidence_value),
                exec_time,
                cpu_energy,
                gpu_energy,
                ram_energy,
                total_energy,
                emissions_value,
                carbon_intensity,
                gpu_snap,
            ) = run_with_energy_tracking(self._infer, image_tensor)

            total_action_time += exec_time

            correct = predicted_label == true_label

            if correct:
                correct_count += 1
                class_correct[true_label] += 1
            else:
                wrong_count += 1

            class_total[true_label] += 1

            true_ids.append(true_label)
            pred_ids.append(predicted_label)

            msg = (
                f"Classified MNIST image using image-text similarity. "
                f"True: {true_label}, Predicted: {predicted_label}, "
                f"Confidence: {confidence_value:.6f}, Correct: {correct}"
            )

            row = self._build_log_row(
                task_id=task_id,
                agent_step="[AGENT ACTION]",
                message=msg,
                step_time=exec_time,
                true_label=true_label,
                predicted_label=predicted_label,
                logits=logits,
                cpu_energy=cpu_energy,
                gpu_energy=gpu_energy,
                ram_energy=ram_energy,
                total_energy=total_energy,
                emissions_value=emissions_value,
                carbon_intensity=carbon_intensity,
                gpu_metrics=gpu_snap,
            )

            all_logs.append(row)
            self._print_log(row)

            if (task_index + 1) % 500 == 0 or (task_index + 1) == total_tasks:
                print(f"  Completed {task_index + 1}/{total_tasks} samples")

        overall_runtime = time.perf_counter() - overall_start

        if SKLEARN_AVAILABLE:
            accuracy = accuracy_score(true_ids, pred_ids)
            prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
                true_ids,
                pred_ids,
                average="weighted",
                zero_division=0
            )
        else:
            accuracy = correct_count / total_tasks
            prec_w = None
            rec_w = None
            f1_w = None

        for row in all_logs:
            row["model_accuracy"] = accuracy
            row["model_precision_weighted"] = prec_w
            row["model_recall_weighted"] = rec_w
            row["model_f1_weighted"] = f1_w

        self._save_logs(all_logs, log_file)

        per_class_accuracy = {
            d: round(100 * class_correct[d] / class_total[d], 2)
            if class_total[d] > 0 else 0.0
            for d in range(10)
        }

        print("\n" + "=" * 60)
        print("Tiny VLM inference logging complete")
        print("=" * 60)
        print(f"Rows   : {len(all_logs)}")
        print(f"Columns: {len(all_logs[0]) if all_logs else 0}")
        print("\nFinal metrics:")
        print(f"  Accuracy           : {accuracy:.4f}")

        if prec_w is not None:
            print(f"  Weighted Precision : {prec_w:.4f}")
            print(f"  Weighted Recall    : {rec_w:.4f}")
            print(f"  Weighted F1        : {f1_w:.4f}")

        return {
            "status": "success",
            "task": "MNIST classification with full Tiny VLM agent log",
            "classifier": self.classifier_name,
            "total_images_classified": total_tasks,
            "correct_predictions": correct_count,
            "wrong_predictions": wrong_count,
            "test_accuracy_percent": round(accuracy * 100, 2),
            "weighted_precision": round(prec_w, 4) if prec_w is not None else None,
            "weighted_recall": round(rec_w, 4) if rec_w is not None else None,
            "weighted_f1": round(f1_w, 4) if f1_w is not None else None,
            "total_action_time_seconds": format(total_action_time, ".20f"),
            "average_action_time_per_image_seconds": format(
                total_action_time / total_tasks,
                ".20f"
            ),
            "overall_runtime_seconds": format(overall_runtime, ".20f"),
            "log_file": log_file.replace(".csv", ".xlsx") if SAVE_EXCEL else log_file,
            "per_class_accuracy_percent": per_class_accuracy,
        }


# ============================================================
# 11. Main
# ============================================================

def checkpoint_is_compatible(model_path):
    """
    Prevents crashes when an old checkpoint was created with a different model.
    """
    if not os.path.exists(model_path):
        return False

    try:
        checkpoint = torch.load(model_path, map_location="cpu")
        required_keys = {
            "model_state_dict",
            "vocab_size",
            "token_to_id",
            "id_to_token",
            "seq_len",
            "embed_dim",
            "text_hidden_dim",
            "num_classes",
        }

        if not required_keys.issubset(set(checkpoint.keys())):
            return False

        test_model = TinyVLM(
            vocab_size=checkpoint["vocab_size"],
            embed_dim=checkpoint.get("embed_dim", 64),
            text_hidden_dim=checkpoint.get("text_hidden_dim", 64),
        )

        test_model.load_state_dict(checkpoint["model_state_dict"])
        return True

    except Exception as e:
        print(f"Incompatible TinyVLM checkpoint detected: {e}")
        return False


if __name__ == "__main__":
    model_path = "tiny_vlm_mnist_classifier.pth"

    if not checkpoint_is_compatible(model_path):
        if os.path.exists(model_path):
            backup_path = model_path + ".old"
            os.replace(model_path, backup_path)
            print(f"Old incompatible checkpoint moved to: {backup_path}")

        train_tiny_vlm(model_path=model_path)
    else:
        print("Saved compatible Tiny VLM classifier found. Skipping training.")

    agent = MNISTTinyVLMAgent(model_path=model_path)

    output = agent.run(
        task="Classify all 10000 MNIST test images using Tiny VLM",
        log_file="mnist_agent_tiny_vlm_full_log.csv",
    )

    print("\n[AGENT FINAL OUTPUT]")
    print(output)

    try:
        if NVML_AVAILABLE:
            pynvml.nvmlShutdown()
    except Exception:
        pass